# Background

## Initialize

In [ ]:
# -*- coding: utf-8 -*-
"""CNN_experiment

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1JBc7Vbzs4INUf8WtQEHVz95W04UqvZQp

# Background

## Initialize
"""

"""
BTC Price Prediction with CNN Models
=====================================
A clean, modular notebook for predicting Bitcoin price movements using CNN models.

Target: Predict if BTC price will go up/down by p% in next N bars
- Up p% or more: +1
- Down p% or more: -1
- Otherwise: 0 (neutral)
"""

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import seaborn as sns
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization, LSTM, GRU, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

CONFIG = {
    # Data parameters
    'ticker': 'BTC-USD',
    'start_date': '2015-01-01',
    'end_date': '2024-11-01',

    # Target parameters
    'target_pct': 1.0,      # Percentage threshold (1.0 = 1%)
    'target_bars': 5,       # Look ahead N bars

    # Model parameters
    'sequence_length': 60,  # Number of time steps for CNN input
    'train_split': 0.7,     # 70% training
    'val_split': 0.15,      # 15% validation
    'test_split': 0.15,     # 15% test

    # Training parameters
    'epochs': 50,
    'batch_size': 32,
    'learning_rate': 0.001,
}

print("="*80)
print("BTC PRICE PREDICTION WITH CNN MODELS")
print("="*80)
print(f"Ticker: {CONFIG['ticker']}")
print(f"Date Range: {CONFIG['start_date']} to {CONFIG['end_date']}")
print(f"Target: {CONFIG['target_pct']}% price change in {CONFIG['target_bars']} bars")
print(f"Sequence Length: {CONFIG['sequence_length']} bars")
print("="*80)

## PREPARE DATA

In [ ]:
# ============================================================================
# 1. DATA PREPARATION
# ============================================================================

def generate_features(df: pd.DataFrame, price_change_threshold=0.02):
    df = df.copy()
    df.columns = df.columns.str.lower()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)

    # ==================== PRICE-BASED FEATURES ====================

    price_features = {}

    # Returns and log returns
    for period in [1, 3, 6, 12, 24]:
        return_period = f'return_{period}h'
        log_return_period = f'log_return_{period}h'
        price_features[return_period] = df['close'].pct_change(period)
        price_features[log_return_period] = np.log(df['close'] / df['close'].shift(period))

    # Price momentum
    for period in [3, 6, 12, 24, 48]:
        momentum_period = f'momentum_{period}h'
        momentum_pct_period = f'momentum_pct_{period}h'
        price_features[momentum_period] = df['close'] - df['close'].shift(period)
        price_features[momentum_pct_period] = (df['close'] - df['close'].shift(period)) / df['close'].shift(period)

    # High-Low spread
    price_features['hl_spread'] = df['high'] - df['low']
    price_features['hl_spread_pct'] = (df['high'] - df['low']) / df['close']
    price_features['close_position'] = (df['close'] - df['low']) / (df['high'] - df['low'] + 1e-10)

    df = pd.concat([df, pd.DataFrame(price_features, index=df.index)], axis=1)

    # ==================== MOVING AVERAGES ====================

    ma_features = {}
    ma_periods = [5, 10, 20, 50, 100, 200]

    for period in ma_periods:
        ma_features[f'sma_{period}'] = df['close'].rolling(window=period).mean()
        ma_features[f'ema_{period}'] = df['close'].ewm(span=period, adjust=False).mean()

    df = pd.concat([df, pd.DataFrame(ma_features, index=df.index)], axis=1)

    # Price ratios (need MA columns to exist first)
    ratio_features = {}
    for period in ma_periods:
        ratio_features[f'price_to_sma_{period}'] = df['close'] / df[f'sma_{period}']
        ratio_features[f'price_to_ema_{period}'] = df['close'] / df[f'ema_{period}']

    # Moving average crossovers
    ratio_features['sma_cross_5_20'] = df['sma_5'] - df['sma_20']
    ratio_features['sma_cross_10_50'] = df['sma_10'] - df['sma_50']
    ratio_features['ema_cross_5_20'] = df['ema_5'] - df['ema_20']

    df = pd.concat([df, pd.DataFrame(ratio_features, index=df.index)], axis=1)

    # ==================== VOLATILITY FEATURES ====================

    vol_features = {}

    # Rolling standard deviation
    for period in [5, 10, 20, 50]:
        vol_features[f'volatility_{period}h'] = df['return_1h'].rolling(window=period).std()
        vol_features[f'price_std_{period}h'] = df['close'].rolling(window=period).std()

    # Bollinger Bands
    for period in [20, 50]:
        rolling_mean = df['close'].rolling(window=period).mean()
        rolling_std = df['close'].rolling(window=period).std()
        vol_features[f'bb_upper_{period}'] = rolling_mean + (rolling_std * 2)
        vol_features[f'bb_lower_{period}'] = rolling_mean - (rolling_std * 2)
        vol_features[f'bb_width_{period}'] = (vol_features[f'bb_upper_{period}'] - vol_features[f'bb_lower_{period}']) / rolling_mean
        vol_features[f'bb_position_{period}'] = (df['close'] - vol_features[f'bb_lower_{period}']) / (vol_features[f'bb_upper_{period}'] - vol_features[f'bb_lower_{period}'] + 1e-10)

    # Average True Range (ATR)
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)

    for period in [14, 28]:
        vol_features[f'atr_{period}'] = true_range.rolling(window=period).mean()
        vol_features[f'atr_pct_{period}'] = vol_features[f'atr_{period}'] / df['close']

    df = pd.concat([df, pd.DataFrame(vol_features, index=df.index)], axis=1)

    # ==================== MOMENTUM INDICATORS ====================

    momentum_features = {}

    # RSI (Relative Strength Index)
    delta = df['close'].diff()
    for period in [14, 28]:
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / (loss + 1e-10)
        momentum_features[f'rsi_{period}'] = 100 - (100 / (1 + rs))

    # MACD (Moving Average Convergence Divergence)
    exp1 = df['close'].ewm(span=12, adjust=False).mean()
    exp2 = df['close'].ewm(span=26, adjust=False).mean()
    momentum_features['macd'] = exp1 - exp2
    momentum_features['macd_signal'] = momentum_features['macd'].ewm(span=9, adjust=False).mean()
    momentum_features['macd_diff'] = momentum_features['macd'] - momentum_features['macd_signal']

    # Stochastic Oscillator
    for period in [14, 28]:
        low_min = df['low'].rolling(window=period).min()
        high_max = df['high'].rolling(window=period).max()
        momentum_features[f'stoch_{period}'] = 100 * (df['close'] - low_min) / (high_max - low_min + 1e-10)

    df = pd.concat([df, pd.DataFrame(momentum_features, index=df.index)], axis=1)

    # Stochastic signals (need stoch columns first)
    stoch_signals = {}
    for period in [14, 28]:
        stoch_signals[f'stoch_signal_{period}'] = df[f'stoch_{period}'].rolling(window=3).mean()

    # Rate of Change (ROC)
    for period in [6, 12, 24]:
        stoch_signals[f'roc_{period}'] = ((df['close'] - df['close'].shift(period)) / df['close'].shift(period)) * 100

    df = pd.concat([df, pd.DataFrame(stoch_signals, index=df.index)], axis=1)

    # ==================== VOLUME FEATURES ====================

    volume_features = {}

    # Volume changes
    for period in [1, 3, 6, 12, 24]:
        volume_features[f'volume_change_{period}h'] = df['volume'].pct_change(period)

    # Volume moving averages
    for period in [5, 10, 20, 50]:
        volume_features[f'volume_sma_{period}'] = df['volume'].rolling(window=period).mean()

    df = pd.concat([df, pd.DataFrame(volume_features, index=df.index)], axis=1)

    # Volume ratios (need volume_sma columns first)
    volume_ratios = {}
    for period in [5, 10, 20, 50]:
        volume_ratios[f'volume_ratio_{period}'] = df['volume'] / df[f'volume_sma_{period}']

    # On-Balance Volume (OBV)
    volume_ratios['obv'] = (np.sign(df['close'].diff()) * df['volume']).fillna(0).cumsum()
    volume_ratios['obv_ema_10'] = volume_ratios['obv'].ewm(span=10, adjust=False).mean()
    volume_ratios['obv_ema_20'] = volume_ratios['obv'].ewm(span=20, adjust=False).mean()

    # Volume-Price Trend
    volume_ratios['vpt'] = (df['volume'] * ((df['close'] - df['close'].shift(1)) / df['close'].shift(1))).fillna(0).cumsum()

    # Money Flow Index (MFI)
    typical_price = (df['high'] + df['low'] + df['close']) / 3
    money_flow = typical_price * df['volume']
    for period in [14, 28]:
        positive_flow = money_flow.where(typical_price > typical_price.shift(1), 0).rolling(window=period).sum()
        negative_flow = money_flow.where(typical_price < typical_price.shift(1), 0).rolling(window=period).sum()
        mfi_ratio = positive_flow / (negative_flow + 1e-10)
        volume_ratios[f'mfi_{period}'] = 100 - (100 / (1 + mfi_ratio))

    df = pd.concat([df, pd.DataFrame(volume_ratios, index=df.index)], axis=1)

    # ==================== PATTERN FEATURES ====================

    pattern_features = {}
    pattern_features['body'] = df['close'] - df['open']
    pattern_features['body_pct'] = pattern_features['body'] / df['open']
    pattern_features['upper_shadow'] = df['high'] - df[['open', 'close']].max(axis=1)
    pattern_features['lower_shadow'] = df[['open', 'close']].min(axis=1) - df['low']
    pattern_features['shadow_ratio'] = (pattern_features['upper_shadow'] + pattern_features['lower_shadow']) / (np.abs(pattern_features['body']) + 1e-10)
    pattern_features['is_doji'] = (np.abs(pattern_features['body']) / (df['high'] - df['low'] + 1e-10) < 0.1).astype(int)
    pattern_features['is_hammer'] = ((pattern_features['lower_shadow'] > 2 * np.abs(pattern_features['body'])) &
                                    (pattern_features['upper_shadow'] < np.abs(pattern_features['body']))).astype(int)

    df = pd.concat([df, pd.DataFrame(pattern_features, index=df.index)], axis=1)

    # ==================== TIME-BASED FEATURES ====================

    time_features = {}
    time_features['hour'] = df['timestamp'].dt.hour
    time_features['day_of_week'] = df['timestamp'].dt.dayofweek
    time_features['day_of_month'] = df['timestamp'].dt.day
    time_features['month'] = df['timestamp'].dt.month

    # Cyclical encoding
    time_features['hour_sin'] = np.sin(2 * np.pi * time_features['hour'] / 24)
    time_features['hour_cos'] = np.cos(2 * np.pi * time_features['hour'] / 24)
    time_features['day_sin'] = np.sin(2 * np.pi * time_features['day_of_week'] / 7)
    time_features['day_cos'] = np.cos(2 * np.pi * time_features['day_of_week'] / 7)

    df = pd.concat([df, pd.DataFrame(time_features, index=df.index)], axis=1)

    # ==================== STATISTICAL FEATURES ====================

    stat_features = {}

    # Rolling statistics
    for period in [10, 20, 50]:
        stat_features[f'price_skew_{period}'] = df['close'].rolling(window=period).skew()
        stat_features[f'price_kurt_{period}'] = df['close'].rolling(window=period).kurt()
        stat_features[f'volume_skew_{period}'] = df['volume'].rolling(window=period).skew()

    # Price percentile in rolling window
    for period in [20, 50, 100]:
        stat_features[f'price_percentile_{period}'] = df['close'].rolling(window=period).apply(
            lambda x: pd.Series(x).rank(pct=True).iloc[-1] if len(x) > 0 else np.nan
        )

    df = pd.concat([df, pd.DataFrame(stat_features, index=df.index)], axis=1)

    # ==================== CLEAN DATA ====================

    df = df.replace([np.inf, -np.inf], np.nan)
    exclude_cols = ['timestamp', 'target', 'price_change_pct', 'open', 'high', 'low', 'close', 'volume']
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    df[feature_cols] = df[feature_cols].fillna(method='ffill').fillna(method='bfill')
    df = df.dropna()

    # ==================== SCALE DATA =====================

    df, scaler = scale_data(df)

    # ==================== TRAIN/VAL/TEST SPLIT ====================

    test_start_date = df['timestamp'].max() - pd.DateOffset(months=1)
    val_start_date = df['timestamp'].max() - pd.DateOffset(months=2)

    train_data = df[df['timestamp'] < val_start_date]
    val_data = df[(df['timestamp'] >= val_start_date) & (df['timestamp'] < test_start_date)]
    test_data = df[df['timestamp'] >= test_start_date]

    X_train, y_train = train_data[feature_cols], train_data['target']
    X_val, y_val = val_data[feature_cols], val_data['target']
    X_test, y_test = test_data[feature_cols], test_data['target']

    print(f"\n{'='*70}")
    print(f"CRYPTO FEATURE ENGINEERING SUMMARY")
    print(f"{'='*70}")
    print(f"Total features created: {len(feature_cols)}")
    print(f"Price change threshold: {price_change_threshold*100:.1f}%")
    print(f"\nData splits:")
    print(f"  Training:   {len(X_train):,} samples ({len(X_train)/len(df)*100:.1f}%)")
    print(f"  Validation: {len(X_val):,} samples ({len(X_val)/len(df)*100:.1f}%)")
    print(f"  Test:       {len(X_test):,} samples ({len(X_test)/len(df)*100:.1f}%)")
    print(f"\nTarget distribution (Training):")
    print(f"  Class 0 (No rise): {(y_train==0).sum():,} ({(y_train==0).sum()/len(y_train)*100:.1f}%)")
    print(f"  Class 1 (Rise):    {(y_train==1).sum():,} ({(y_train==1).sum()/len(y_train)*100:.1f}%)")
    print(f"{'='*70}\n")

    return {
        'X_train': X_train, 'y_train': y_train,
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_test, 'y_test': y_test,
        'feature_names': feature_cols,
        'train_data': train_data,
        'val_data': val_data,
        'test_data': test_data,
        'df': df,
        'scaler': scaler
    }

def download_and_prepare_data(config):
    """
    Download BTC data and prepare features with target.

    Returns:
    --------
    dict with keys: df, features, target, feature_names
    """
    print("\n" + "="*80)
    print("STEP 1: DOWNLOADING AND PREPARING DATA")
    print("="*80)

    # Download data
    print(f"\nDownloading {config['ticker']} data...")
    btc_data = yf.download(config['ticker'], start=config['start_date'], end=config['end_date'])
    btc_data.columns = btc_data.columns.get_level_values(0)
    btc_data = btc_data.reset_index()

    # Rename columns to lowercase (yfinance returns: Date, Open, High, Low, Close, Adj Close, Volume)
    btc_data.columns = [col.lower().replace(' ', '_') for col in btc_data.columns]
    btc_data = btc_data.rename(columns={'date': 'timestamp', 'adj_close': 'adj_close'})

    print(f"✓ Downloaded {len(btc_data):,} rows")
    print(f"Columns: {list(btc_data.columns)}")

    # Create target variable
    print("\nCreating target variable...")

    # Calculate future price change
    btc_data['future_close'] = btc_data['close'].shift(-config['target_bars'])
    btc_data['pct_change'] = ((btc_data['future_close'] - btc_data['close']) / btc_data['close']) * 100

    # Create three-class target: 0 (down), 1 (neutral), 2 (up)
    # Using 0,1,2 for compatibility with sparse_categorical_crossentropy
    btc_data['target'] = 1  # neutral (default)
    btc_data.loc[btc_data['pct_change'] >= config['target_pct'], 'target'] = 2  # up
    btc_data.loc[btc_data['pct_change'] <= -config['target_pct'], 'target'] = 0  # down

    # Generate technical features
    print("Generating technical features...")
    generated_results = generate_features(btc_data)
    df = generated_results['df']

    # Clean data
    df = df.dropna()

    # Select feature columns
    exclude_cols = ['timestamp', 'target', 'open', 'high', 'low', 'close', 'volume', 'adj_close',
                    'future_close', 'pct_change']
    feature_cols = [col for col in df.columns if col not in exclude_cols]

    features = df[feature_cols].values
    target = df['target'].values

    print(f"✓ Generated {len(feature_cols)} features")
    print(f"\nTarget distribution:")
    print(f"  Down (0):     {(target == 0).sum():,} ({(target == 0).sum()/len(target)*100:.1f}%)")
    print(f"  Neutral (1):  {(target == 1).sum():,} ({(target == 1).sum()/len(target)*100:.1f}%)")
    print(f"  Up (2):       {(target == 2).sum():,} ({(target == 2).sum()/len(target)*100:.1f}%)")

    return {
        'df': df,
        'features': features,
        'target': target,
        'feature_names': feature_cols,
        'generated_results': generated_results
    }


def create_sequences(features, target, sequence_length):
    """
    Create sequences for CNN input.

    Parameters:
    -----------
    features : np.array
        Feature matrix (samples, features)
    target : np.array
        Target vector
    sequence_length : int
        Number of time steps in each sequence

    Returns:
    --------
    X_sequences, y_sequences : np.arrays
    """
    X_sequences = []
    y_sequences = []

    for i in range(len(features) - sequence_length):
        X_sequences.append(features[i:(i + sequence_length)])
        y_sequences.append(target[i + sequence_length])

    return np.array(X_sequences), np.array(y_sequences)

def scale_data(df, scaler = None):
    if scaler is None:
        scaler = MinMaxScaler()
    float_cols = [col for col in df.columns if df[col].dtype in ['float64', 'float32']]
    df[float_cols] = scaler.fit_transform(df[float_cols])
    return df, scaler

def prepare_train_val_test_data(data_dict, config):
    """
    Split data into train/val/test sets and create sequences.

    Returns:
    --------
    dict with train/val/test data
    """
    print("\n" + "="*80)
    print("STEP 2: CREATING SEQUENCES AND SPLITTING DATA")
    print("="*80)

    features = data_dict['features']
    target = data_dict['target']

    # Create sequences
    print(f"Creating sequences (length={config['sequence_length']})...")
    X, y = create_sequences(features, target, config['sequence_length'])

    # Split into train/val/test
    n_samples = len(X)
    train_size = int(n_samples * config['train_split'])
    val_size = int(n_samples * config['val_split'])

    X_train = X[:train_size]
    y_train = y[:train_size]

    X_val = X[train_size:train_size + val_size]
    y_val = y[train_size:train_size + val_size]

    X_test = X[train_size + val_size:]
    y_test = y[train_size + val_size:]

    print(f"\n✓ Created {len(X):,} sequences")
    print(f"\nData splits:")
    print(f"  Training:   {len(X_train):,} sequences ({len(X_train)/len(X)*100:.1f}%)")
    print(f"  Validation: {len(X_val):,} sequences ({len(X_val)/len(X)*100:.1f}%)")
    print(f"  Test:       {len(X_test):,} sequences ({len(X_test)/len(X)*100:.1f}%)")

    print(f"\nInput shape: {X_train.shape}")
    print(f"  - Samples: {X_train.shape[0]}")
    print(f"  - Time steps: {X_train.shape[1]}")
    print(f"  - Features: {X_train.shape[2]}")

    return {
        'X_train': X_train, 'y_train': y_train,
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_test, 'y_test': y_test
    }



## CREATE MODELS

In [ ]:
# ============================================================================
# 2. MODEL CREATION
# ============================================================================


def create_models(input_shape, num_classes=3):
    """
    Create multiple model architectures: CNN, LSTM, GRU, and traditional ML.

    Parameters:
    -----------
    input_shape : tuple
        (time_steps, features) - for neural networks
    num_classes : int
        Number of output classes (3 for up/neutral/down)

    Returns:
    --------
    dict of models (both neural networks and traditional ML)
    """
    print("\n" + "="*80)
    print("STEP 3: CREATING MODELS")
    print("="*80)

    models = {}

    # ========== NEURAL NETWORK MODELS (require 3D input) ==========
    print("\n--- Neural Network Models ---")

    # Model 1: Simple CNN
    print("\nCreating Model 1: Simple CNN...")
    model1 = Sequential([
        Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=input_shape),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(50, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ], name='Simple_CNN')

    model1.compile(
        optimizer=Adam(learning_rate=CONFIG['learning_rate']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    models['simple_cnn'] = model1
    print(f"✓ Simple CNN: {model1.count_params():,} parameters")

    # Model 2: Deep CNN
    print("\nCreating Model 2: Deep CNN...")
    model2 = Sequential([
        Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Conv1D(filters=64, kernel_size=3, activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(100, activation='relu'),
        Dropout(0.5),
        Dense(50, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ], name='Deep_CNN')

    model2.compile(
        optimizer=Adam(learning_rate=CONFIG['learning_rate']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    models['deep_cnn'] = model2
    print(f"✓ Deep CNN: {model2.count_params():,} parameters")

    # Model 3: Wide CNN (different kernel sizes)
    print("\nCreating Model 3: Wide CNN...")
    model3 = Sequential([
        Conv1D(filters=32, kernel_size=5, activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Conv1D(filters=64, kernel_size=3, activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(100, activation='relu'),
        Dropout(0.4),
        Dense(num_classes, activation='softmax')
    ], name='Wide_CNN')

    model3.compile(
        optimizer=Adam(learning_rate=CONFIG['learning_rate']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    models['wide_cnn'] = model3
    print(f"✓ Wide CNN: {model3.count_params():,} parameters")

    # Model 4: Simple LSTM
    print("\nCreating Model 4: Simple LSTM...")
    model4 = Sequential([
        LSTM(64, activation='tanh', input_shape=input_shape, return_sequences=False),
        Dropout(0.3),
        Dense(50, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ], name='Simple_LSTM')

    model4.compile(
        optimizer=Adam(learning_rate=CONFIG['learning_rate']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    models['simple_lstm'] = model4
    print(f"✓ Simple LSTM: {model4.count_params():,} parameters")

    # Model 5: Bidirectional LSTM
    print("\nCreating Model 5: Bidirectional LSTM...")
    model5 = Sequential([
        Bidirectional(LSTM(64, activation='tanh', return_sequences=True), input_shape=input_shape),
        Dropout(0.3),
        Bidirectional(LSTM(32, activation='tanh')),
        Dropout(0.3),
        Dense(50, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ], name='BiLSTM')

    model5.compile(
        optimizer=Adam(learning_rate=CONFIG['learning_rate']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    models['bilstm'] = model5
    print(f"✓ Bidirectional LSTM: {model5.count_params():,} parameters")

    # Model 6: Simple GRU
    print("\nCreating Model 6: Simple GRU...")
    model6 = Sequential([
        GRU(64, activation='tanh', input_shape=input_shape, return_sequences=False),
        Dropout(0.3),
        Dense(50, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ], name='Simple_GRU')

    model6.compile(
        optimizer=Adam(learning_rate=CONFIG['learning_rate']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    models['simple_gru'] = model6
    print(f"✓ Simple GRU: {model6.count_params():,} parameters")

    # Model 7: CNN-LSTM Hybrid
    print("\nCreating Model 7: CNN-LSTM Hybrid...")
    model7 = Sequential([
        Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        LSTM(64, activation='tanh', return_sequences=False),
        Dropout(0.4),
        Dense(50, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ], name='CNN_LSTM_Hybrid')

    model7.compile(
        optimizer=Adam(learning_rate=CONFIG['learning_rate']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    models['cnn_lstm'] = model7
    print(f"✓ CNN-LSTM Hybrid: {model7.count_params():,} parameters")

    # ========== TRADITIONAL ML MODELS (require 2D input) ==========
    print("\n--- Traditional ML Models ---")
    print("Note: These models will be trained on flattened sequences")

    # Model 8: Logistic Regression
    print("\nCreating Model 8: Logistic Regression...")
    model8 = LogisticRegression(
        max_iter=1000,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    )
    models['logistic_regression'] = model8
    print("✓ Logistic Regression created")

    # Model 9: Random Forest
    print("\nCreating Model 9: Random Forest...")
    model9 = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    )
    models['random_forest'] = model9
    print("✓ Random Forest created")

    # Model 10: Gradient Boosting
    #print("\nCreating Model 10: Gradient Boosting...")
    #model10 = GradientBoostingClassifier(
    #    n_estimators=100,
    #    max_depth=5,
    #    learning_rate=0.1,
    #    random_state=42
    #)
    #models['gradient_boosting'] = model10
    #print("✓ Gradient Boosting created")

    # Model 11: XGBoost (if available)
    if XGBOOST_AVAILABLE:
        print("\nCreating Model 11: XGBoost...")
        model11 = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.1,
            random_state=42,
            n_jobs=-1,
            eval_metric='mlogloss'
        )
        models['xgboost'] = model11
        print("✓ XGBoost created")
    else:
        print("\n⚠ XGBoost not available (install with: pip install xgboost)")

    # Model 12: LightGBM (if available)
    if LIGHTGBM_AVAILABLE:
        print("\nCreating Model 12: LightGBM...")
        model12 = lgb.LGBMClassifier(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.1,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
        models['lightgbm'] = model12
        print("✓ LightGBM created")
    else:
        print("\n⚠ LightGBM not available (install with: pip install lightgbm)")

    # Model 13: SVM (for smaller datasets)
    #print("\nCreating Model 13: SVM...")
    #model13 = SVC(
    #    kernel='rbf',
    #    C=1.0,
    #    probability=True,  # Enable probability estimates
    #    random_state=42,
    #    class_weight='balanced'
    #)
    #models['svm'] = model13
    #print("✓ SVM created")

    print(f"\n✓ Created {len(models)} models")
    print(f"  - Neural Networks: {sum(1 for k in models.keys() if k in ['simple_cnn', 'deep_cnn', 'wide_cnn', 'simple_lstm', 'bilstm', 'simple_gru', 'cnn_lstm'])}")
    print(f"  - Traditional ML: {sum(1 for k in models.keys() if k not in ['simple_cnn', 'deep_cnn', 'wide_cnn', 'simple_lstm', 'bilstm', 'simple_gru', 'cnn_lstm'])}")

    return models

In [ ]:
import joblib
def save_models(models, scaler):
    # Save each model
    for name, model in models.items():
        filename = f"{name}.joblib"
        joblib.dump(model, filename)
        print(f"✓ Saved {name}")
    # Save scaler
    if scaler is not None:
        scaler_path = "scaler.joblib"
        joblib.dump(scaler, scaler_path)
        print(f"✓ Saved scaler")


def load_models():
    models = {}
    scaler = None
    # List joblib files in folder
    files = [f for f in os.listdir() if f.endswith('.joblib')]
    scaler = [f for f in files if 'scaler' in f]
    if scaler:
        scaler_path = scaler[0]
        scaler = joblib.load(scaler_path)
        print(f"✓ Loaded scaler")
    # Load each model
    models_to_load = [f for f in files if 'scaler' not in f]
    for name in models_to_load:
        filename = f"{name}.joblib"
        model = joblib.load(filename)
        models[name] = model
        print(f"✓ Loaded {name}")
    return models, scaler

## TRAIN MODELS

In [ ]:
# ============================================================================
# 3. TRAINING
# ============================================================================


def train_models(models, data_dict, config):
    """
    Train all models.

    Returns:
    --------
    dict with training histories
    """
    print("\n" + "="*80)
    print("STEP 4: TRAINING MODELS")
    print("="*80)

    X_train = data_dict['X_train']
    y_train = data_dict['y_train']
    X_val = data_dict['X_val']
    y_val = data_dict['y_val']

    # Callbacks
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=0
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=0
    )

    histories = {}

    # Separate neural networks from traditional ML models
    neural_models = ['simple_cnn', 'deep_cnn', 'wide_cnn', 'simple_lstm', 'bilstm', 'simple_gru', 'cnn_lstm']

    for model_name, model in models.items():
        print(f"\nTraining {model_name}...")
        print("-" * 60)

        # Check if it's a neural network (Keras) or traditional ML (sklearn)
        if model_name in neural_models or hasattr(model, 'layers'):
            # Neural Network - use Keras training
            history = model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=config['epochs'],
                batch_size=config['batch_size'],
                callbacks=[early_stopping, reduce_lr],
                verbose=1
            )

            histories[model_name] = history

            # Print final metrics
            final_train_acc = history.history['accuracy'][-1]
            final_val_acc = history.history['val_accuracy'][-1]
            print(f"\n✓ Training complete:")
            print(f"  Final train accuracy: {final_train_acc:.4f}")
            print(f"  Final val accuracy:   {final_val_acc:.4f}")
        else:
            # Traditional ML - flatten sequences and use sklearn training
            print("  (Flattening sequences for traditional ML model...)")
            X_train_flat = X_train.reshape(X_train.shape[0], -1)
            X_val_flat = X_val.reshape(X_val.shape[0], -1)

            # Train model
            model.fit(X_train_flat, y_train)

            # Calculate accuracy on train and validation
            train_acc = model.score(X_train_flat, y_train)
            val_acc = model.score(X_val_flat, y_val)

            # Store as simple dict (not Keras history)
            histories[model_name] = {
                'train_accuracy': train_acc,
                'val_accuracy': val_acc
            }

            print(f"\n✓ Training complete:")
            print(f"  Train accuracy: {train_acc:.4f}")
            print(f"  Val accuracy:   {val_acc:.4f}")

    return histories

## TEST MODELS

In [ ]:
# ============================================================================
# 4. TESTING & EVALUATION
# ============================================================================


def test_models(models, data_dict):
    """
    Test all models and print detailed metrics.

    Returns:
    --------
    dict with test results
    """
    print("\n" + "="*80)
    print("STEP 5: TESTING MODELS")
    print("="*80)

    X_test = data_dict['X_test']
    y_test = data_dict['y_test']

    # Separate neural networks from traditional ML models
    neural_models = ['simple_cnn', 'deep_cnn', 'wide_cnn', 'simple_lstm', 'bilstm', 'simple_gru', 'cnn_lstm']

    results = {}

    for model_name, model in models.items():
        print(f"\n{'='*60}")
        print(f"Testing {model_name.upper()}")
        print(f"{'='*60}")

        # Get predictions based on model type
        if model_name in neural_models or hasattr(model, 'layers'):
            # Neural Network - use Keras predict with verbose
            y_pred_proba = model.predict(X_test, verbose=0)
            y_pred = np.argmax(y_pred_proba, axis=1)
        else:
            # Traditional ML - flatten and predict
            print("  (Flattening sequences for traditional ML model...)")
            X_test_flat = X_test.reshape(X_test.shape[0], -1)

            # Get predictions
            y_pred = model.predict(X_test_flat)

            # Get probabilities if available
            if hasattr(model, 'predict_proba'):
                y_pred_proba = model.predict_proba(X_test_flat)
            else:
                # Create dummy probabilities for models without predict_proba
                y_pred_proba = np.zeros((len(y_pred), 3))
                y_pred_proba[np.arange(len(y_pred)), y_pred] = 1.0

        # Labels are already 0, 1, 2 (down, neutral, up)
        y_pred_labeled = y_pred
        y_test_labeled = y_test

        # Metrics
        accuracy = accuracy_score(y_test, y_pred)

        # Per-class metrics
        precision = precision_score(y_test, y_pred, average=None, zero_division=0)
        recall = recall_score(y_test, y_pred, average=None, zero_division=0)
        f1 = f1_score(y_test, y_pred, average=None, zero_division=0)

        # Weighted averages
        precision_weighted = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall_weighted = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)

        # Store results
        results[model_name] = {
            'y_pred': y_pred,
            'y_pred_labeled': y_pred_labeled,
            'y_pred_proba': y_pred_proba,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'precision_weighted': precision_weighted,
            'recall_weighted': recall_weighted,
            'f1_weighted': f1_weighted,
            'confusion_matrix': cm
        }

        # Print metrics
        print(f"\nOverall Metrics:")
        print(f"  Accuracy:  {accuracy:.4f}")
        print(f"  Precision: {precision_weighted:.4f} (weighted)")
        print(f"  Recall:    {recall_weighted:.4f} (weighted)")
        print(f"  F1-Score:  {f1_weighted:.4f} (weighted)")

        print(f"\nPer-Class Metrics:")
        class_names = ['Down (0)', 'Neutral (1)', 'Up (2)']
        for i, class_name in enumerate(class_names):
            print(f"  {class_name}:")
            print(f"    Precision: {precision[i]:.4f}")
            print(f"    Recall:    {recall[i]:.4f}")
            print(f"    F1-Score:  {f1[i]:.4f}")

        print(f"\nConfusion Matrix:")
        print(f"  Predicted →")
        print(f"  Actual ↓     Down    Neutral    Up")
        for i, class_name in enumerate(['Down', 'Neutral', 'Up']):
            print(f"  {class_name:8s}  {cm[i][0]:6d}  {cm[i][1]:6d}  {cm[i][2]:6d}")

        # Classification report
        print(f"\nDetailed Classification Report:")
        print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

    return results

## VISUALIZATION

In [ ]:
# ============================================================================
# 5. VISUALIZATION
# ============================================================================


def visualize_results(histories, results, models):
    """
    Create comprehensive visualizations.
    """
    print("\n" + "="*80)
    print("STEP 6: VISUALIZING RESULTS")
    print("="*80)

    # Separate neural networks from traditional ML models
    neural_models = ['simple_cnn', 'deep_cnn', 'wide_cnn', 'simple_lstm', 'bilstm', 'simple_gru', 'cnn_lstm']

    # Filter only neural network histories (they have training curves)
    neural_histories = {k: v for k, v in histories.items() if k in neural_models or hasattr(v, 'history')}
    n_neural = len(neural_histories)

    if n_neural > 0:
        # Figure 1: Training History (only for neural networks)
        fig1, axes1 = plt.subplots(2, n_neural, figsize=(6*n_neural, 10))
        if n_neural == 1:
            axes1 = axes1.reshape(-1, 1)

        for idx, (model_name, history) in enumerate(neural_histories.items()):
            # Accuracy
            axes1[0, idx].plot(history.history['accuracy'], label='Train', linewidth=2)
            axes1[0, idx].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
            axes1[0, idx].set_title(f'{model_name.upper()} - Accuracy', fontsize=12, fontweight='bold')
            axes1[0, idx].set_xlabel('Epoch')
            axes1[0, idx].set_ylabel('Accuracy')
            axes1[0, idx].legend()
            axes1[0, idx].grid(True, alpha=0.3)

            # Loss
            axes1[1, idx].plot(history.history['loss'], label='Train', linewidth=2)
            axes1[1, idx].plot(history.history['val_loss'], label='Validation', linewidth=2)
            axes1[1, idx].set_title(f'{model_name.upper()} - Loss', fontsize=12, fontweight='bold')
            axes1[1, idx].set_xlabel('Epoch')
            axes1[1, idx].set_ylabel('Loss')
            axes1[1, idx].legend()
            axes1[1, idx].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('training_history_neural.png', dpi=150, bbox_inches='tight')
        print("\n✓ Saved training_history_neural.png")
        plt.show()
    else:
        print("\n⚠ No neural network models to plot training history")

    # Figure 1b: Training Accuracy Comparison (all models)
    fig1b, ax1b = plt.subplots(1, 1, figsize=(12, 6))

    model_names = []
    train_accs = []
    val_accs = []

    for model_name, history in histories.items():
        model_names.append(model_name)
        if hasattr(history, 'history'):
            # Neural network - get final epoch accuracy
            train_accs.append(history.history['accuracy'][-1])
            val_accs.append(history.history['val_accuracy'][-1])
        else:
            # Traditional ML - get stored accuracy
            train_accs.append(history['train_accuracy'])
            val_accs.append(history['val_accuracy'])

    x = np.arange(len(model_names))
    width = 0.35

    ax1b.bar(x - width/2, train_accs, width, label='Train', alpha=0.8)
    ax1b.bar(x + width/2, val_accs, width, label='Validation', alpha=0.8)
    ax1b.set_xlabel('Model', fontsize=12)
    ax1b.set_ylabel('Accuracy', fontsize=12)
    ax1b.set_title('Training Accuracy Comparison - All Models', fontsize=14, fontweight='bold')
    ax1b.set_xticks(x)
    ax1b.set_xticklabels(model_names, rotation=45, ha='right')
    ax1b.legend()
    ax1b.grid(True, alpha=0.3, axis='y')
    ax1b.set_ylim([0, 1])

    plt.tight_layout()
    plt.savefig('training_accuracy_comparison.png', dpi=150, bbox_inches='tight')
    print("✓ Saved training_accuracy_comparison.png")
    plt.show()

    n_models = len(models)

    # Figure 2: Confusion Matrices
    fig2, axes2 = plt.subplots(1, n_models, figsize=(6*n_models, 5))
    if n_models == 1:
        axes2 = [axes2]

    class_names = ['Down\n(0)', 'Neutral\n(1)', 'Up\n(2)']

    for idx, (model_name, result) in enumerate(results.items()):
        cm = result['confusion_matrix']

        sns.heatmap(
            cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names,
            ax=axes2[idx],
            cbar_kws={'label': 'Count'}
        )
        axes2[idx].set_title(f'{model_name.upper()}\nAccuracy: {result["accuracy"]:.4f}',
                            fontsize=12, fontweight='bold')
        axes2[idx].set_ylabel('Actual')
        axes2[idx].set_xlabel('Predicted')

    plt.tight_layout()
    plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
    print("✓ Saved confusion_matrices.png")
    plt.show()

    # Figure 3: Model Comparison
    fig3, axes3 = plt.subplots(1, 3, figsize=(18, 5))

    model_names = list(results.keys())
    metrics_data = {
        'Accuracy': [results[m]['accuracy'] for m in model_names],
        'Precision': [results[m]['precision_weighted'] for m in model_names],
        'Recall': [results[m]['recall_weighted'] for m in model_names],
        'F1-Score': [results[m]['f1_weighted'] for m in model_names]
    }

    # Overall metrics comparison
    x = np.arange(len(model_names))
    width = 0.2

    for i, (metric_name, values) in enumerate(metrics_data.items()):
        axes3[0].bar(x + i*width, values, width, label=metric_name, alpha=0.8)

    axes3[0].set_xlabel('Model')
    axes3[0].set_ylabel('Score')
    axes3[0].set_title('Overall Metrics Comparison', fontsize=12, fontweight='bold')
    axes3[0].set_xticks(x + width * 1.5)
    axes3[0].set_xticklabels([m.upper() for m in model_names], rotation=45, ha='right')
    axes3[0].legend()
    axes3[0].grid(True, alpha=0.3, axis='y')
    axes3[0].set_ylim([0, 1])

    # Per-class F1 scores
    class_names_short = ['Down', 'Neutral', 'Up']
    for model_name in model_names:
        f1_scores = results[model_name]['f1']
        axes3[1].plot(class_names_short, f1_scores, marker='o', linewidth=2,
                     markersize=8, label=model_name.upper())

    axes3[1].set_xlabel('Class')
    axes3[1].set_ylabel('F1-Score')
    axes3[1].set_title('Per-Class F1-Score Comparison', fontsize=12, fontweight='bold')
    axes3[1].legend()
    axes3[1].grid(True, alpha=0.3)
    axes3[1].set_ylim([0, 1])

    # Best model highlight
    best_model = max(results.items(), key=lambda x: x[1]['accuracy'])
    best_name = best_model[0]
    best_acc = best_model[1]['accuracy']
    best_f1 = best_model[1]['f1_weighted']

    axes3[2].text(0.5, 0.7, 'BEST MODEL', ha='center', va='center',
                 fontsize=20, fontweight='bold', color='darkgreen')
    axes3[2].text(0.5, 0.5, best_name.upper(), ha='center', va='center',
                 fontsize=16, fontweight='bold')
    axes3[2].text(0.5, 0.35, f'Accuracy: {best_acc:.4f}', ha='center', va='center', fontsize=14)
    axes3[2].text(0.5, 0.25, f'F1-Score: {best_f1:.4f}', ha='center', va='center', fontsize=14)
    axes3[2].axis('off')

    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
    print("✓ Saved model_comparison.png")
    plt.show()

    print("\n✓ Visualization complete!")

In [ ]:

def _create_models_comparison_plot(all_results, all_equity_curves, path_suffix):
    """
    Create a comprehensive comparison plot for all tested models.

    Parameters:
    -----------
    all_results : dict
        Dictionary of model_name -> results dict
    all_equity_curves : dict
        Dictionary of model_name -> equity curve DataFrame
    """
    if not all_results:
        print("⚠ No results to plot - all_results is empty")
        return

    if not all_equity_curves:
        print("⚠ No equity curves to plot - all_equity_curves is empty")
        print(f"  Available results keys: {list(all_results.keys())}")
        return

    # Filter valid equity curves
    valid_equity_curves = {}
    for model_name, equity_df in all_equity_curves.items():
        if equity_df is not None and len(equity_df) > 0:
            if 'equity' in equity_df.columns and 'timestamp' in equity_df.columns:
                valid_equity_curves[model_name] = equity_df
            else:
                print(f"⚠ Skipping {model_name}: missing required columns (has: {list(equity_df.columns)})")
        else:
            print(f"⚠ Skipping {model_name}: equity curve is empty or None")

    if not valid_equity_curves:
        print("⚠ No valid equity curves to plot")
        print(f"  Total equity curves: {len(all_equity_curves)}")
        return

    print(f"\nPlotting {len(valid_equity_curves)} model(s)...")

    # Create figure with 2 subplots
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.3)

    # Subplot 1: Equity curves
    ax1 = fig.add_subplot(gs[0])

    # Define colors for models
    colors = plt.cm.tab10(np.linspace(0, 1, len(valid_equity_curves)))

    # Plot each model's equity curve
    for i, (model_name, equity_df) in enumerate(valid_equity_curves.items()):
        # Normalize to percentage return
        initial_capital = all_results[model_name].get('initial_capital', 10000)
        equity_pct = (equity_df['equity'] / initial_capital - 1) * 100

        ax1.plot(equity_df['timestamp'], equity_pct,
                label=model_name.replace('_', ' ').title(),
                linewidth=2, color=colors[i], alpha=0.8)

    ax1.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.3)
    ax1.set_xlabel('Date', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Return (%)', fontsize=12, fontweight='bold')
    ax1.set_title('Model Comparison - Equity Curves', fontsize=14, fontweight='bold')
    ax1.legend(loc='best', fontsize=10, framealpha=0.9)
    ax1.grid(True, alpha=0.3)

    # Format x-axis
    ax1.tick_params(axis='x', rotation=45)

    # Subplot 2: Summary table
    ax2 = fig.add_subplot(gs[1])
    ax2.axis('off')

    # Create summary data
    summary_data = []
    for model_name, results in all_results.items():
        summary_data.append([
            model_name.replace('_', ' ').title(),
            f"${results.get('final_value', 0):,.0f}",
            f"{results.get('total_return_pct', 0):.2f}%",
            f"{results.get('total_trades', 0)}",
            f"{results.get('win_rate', 0):.1f}%",
            f"{results.get('sharpe_ratio', 0):.2f}",
            f"{results.get('max_drawdown', 0):.1f}%"
        ])

    # Sort by total return
    summary_data.sort(key=lambda x: float(x[2].rstrip('%')), reverse=True)

    # Create table
    columns = ['Model', 'Final Value', 'Return', 'Trades', 'Win Rate', 'Sharpe', 'Max DD']

    table = ax2.table(cellText=summary_data, colLabels=columns,
                     cellLoc='center', loc='center',
                     colWidths=[0.20, 0.15, 0.12, 0.10, 0.12, 0.10, 0.12])

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)

    # Style header
    for i in range(len(columns)):
        cell = table[(0, i)]
        cell.set_facecolor('#4CAF50')
        cell.set_text_props(weight='bold', color='white')

    # Style rows - alternate colors
    for i in range(1, len(summary_data) + 1):
        for j in range(len(columns)):
            cell = table[(i, j)]
            if i % 2 == 0:
                cell.set_facecolor('#f0f0f0')
            else:
                cell.set_facecolor('white')

    # Highlight best model (first row after sorting)
    for j in range(len(columns)):
        cell = table[(1, j)]
        cell.set_facecolor('#90EE90')
        cell.set_text_props(weight='bold')

    ax2.set_title('Performance Summary (Sorted by Return)',
                 fontsize=12, fontweight='bold', pad=20)

    plt.tight_layout()

    # Save plot
    filepath = os.path.join(f'models_comparison.png')
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    print(f"\n✓ Comparison plot saved to: {filepath}")

    plt.show()
    plt.close()

def _create_ohlc_with_trades_plot(df_ohlc, trades_df, model_name):
    """
    Create interactive Plotly OHLC candlestick chart with trade entry and exit markers.

    Parameters:
    -----------
    df_ohlc : pd.DataFrame
        DataFrame with OHLC data (must have: timestamp, open, high, low, close, volume)
    trades_df : pd.DataFrame
        DataFrame with trades (must have: entry_time, exit_time, entry_price, exit_price, net_pnl)
    model_name : str
        Name of the model for the title

    Returns:
    --------
    str : Path to saved HTML file
    """
    if df_ohlc is None or len(df_ohlc) == 0:
        print("⚠ No OHLC data to plot")
        return None

    if trades_df is None or len(trades_df) == 0:
        print("⚠ No trades to plot")
        return None

    # Prepare data
    df_plot = df_ohlc.copy()

    # Ensure timestamp column
    if 'timestamp' not in df_plot.columns:
        if isinstance(df_plot.index, pd.DatetimeIndex):
            df_plot['timestamp'] = df_plot.index
        else:
            df_plot['timestamp'] = pd.to_datetime(df_plot.index)
    else:
        df_plot['timestamp'] = pd.to_datetime(df_plot['timestamp'])

    # Ensure lowercase column names
    df_plot.columns = [col.lower() for col in df_plot.columns]

    # Create subplots: candlestick + volume
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        row_heights=[0.7, 0.3],
        subplot_titles=(f'OHLC Chart with Trades - {model_name.replace("_", " ").title()}', 'Volume')
    )

    # Add candlestick chart
    fig.add_trace(
        go.Candlestick(
            x=df_plot['timestamp'],
            open=df_plot['open'],
            high=df_plot['high'],
            low=df_plot['low'],
            close=df_plot['close'],
            name='OHLC',
            increasing_line_color='green',
            decreasing_line_color='red'
        ),
        row=1, col=1
    )

    # Add volume bars
    colors = ['red' if df_plot['close'].iloc[i] < df_plot['open'].iloc[i] else 'green'
              for i in range(len(df_plot))]
    fig.add_trace(
        go.Bar(
            x=df_plot['timestamp'],
            y=df_plot['volume'],
            name='Volume',
            marker_color=colors,
            showlegend=False
        ),
        row=2, col=1
    )

    # Prepare trade markers
    entry_times = []
    entry_prices = []
    exit_win_times = []
    exit_win_prices = []
    exit_loss_times = []
    exit_loss_prices = []

    for _, trade in trades_df.iterrows():
        entry_time = pd.to_datetime(trade['entry_time'])
        exit_time = pd.to_datetime(trade['exit_time'])
        entry_price = trade['entry_price']
        exit_price = trade['exit_price']
        is_win = trade.get('net_pnl', 0) > 0

        entry_times.append(entry_time)
        entry_prices.append(entry_price)

        if is_win:
            exit_win_times.append(exit_time)
            exit_win_prices.append(exit_price)
        else:
            exit_loss_times.append(exit_time)
            exit_loss_prices.append(exit_price)

    # Add entry markers
    fig.add_trace(
        go.Scatter(
            x=entry_times,
            y=entry_prices,
            mode='markers',
            name='Entry',
            marker=dict(
                symbol='triangle-up',
                size=12,
                color='lime',
                line=dict(color='darkgreen', width=1)
            )
        ),
        row=1, col=1
    )

    # Add exit markers (wins)
    if exit_win_times:
        fig.add_trace(
            go.Scatter(
                x=exit_win_times,
                y=exit_win_prices,
                mode='markers',
                name='Exit (Win)',
                marker=dict(
                    symbol='triangle-down',
                    size=12,
                    color='dodgerblue',
                    line=dict(color='darkblue', width=1)
                )
            ),
            row=1, col=1
        )

    # Add exit markers (losses)
    if exit_loss_times:
        fig.add_trace(
            go.Scatter(
                x=exit_loss_times,
                y=exit_loss_prices,
                mode='markers',
                name='Exit (Loss)',
                marker=dict(
                    symbol='triangle-down',
                    size=12,
                    color='red',
                    line=dict(color='darkred', width=1)
                )
            ),
            row=1, col=1
        )

    # Calculate statistics
    total_trades = len(trades_df)
    winning_trades = len(trades_df[trades_df['net_pnl'] > 0])
    win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0

    # Update layout
    fig.update_layout(
        title=dict(
            text=f'{model_name.replace("_", " ").title()}<br><sub>Trades: {total_trades} | Wins: {winning_trades} | Win Rate: {win_rate:.1f}%</sub>',
            x=0.5,
            xanchor='center'
        ),
        xaxis_rangeslider_visible=False,
        height=800,
        hovermode='x unified',
        template='plotly_white'
    )

    # Update axes
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Volume", row=2, col=1)

    # Save as HTML
    filepath = f'ohlc_trades_{model_name}.html'
    fig.write_html(filepath)
    fig.show()
    print(f"  ✓ Interactive OHLC chart saved to: {filepath}")

    return filepath



## BACKTESTING

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional, Union
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from typing import Dict, List, Tuple, Optional, Union, Any
from datetime import datetime, timedelta
from abc import ABC, abstractmethod
import warnings
from pathlib import Path
import json
import os

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")






In [ ]:
class BacktestBase(ABC):
    """
    Abstract base class for all backtesting implementations.

    Provides common functionality for:
    - Trade tracking and analysis
    - Performance metrics calculation
    - Detailed reporting
    - Advanced visualization
    - Export capabilities
    """

    def __init__(
        self,
        initial_capital: float = 10000.0,
        commission: float = 0.001,
        slippage: float = 0.0005,
        **kwargs
    ):
        """
        Initialize base backtest class.

        Parameters:
        -----------
        initial_capital : float
            Starting capital
        commission : float
            Commission per trade as fraction
        slippage : float
            Slippage per trade as fraction
        """
        self.initial_capital = initial_capital
        self.commission = commission
        self.slippage = slippage

        # Results storage
        self.trades = []
        self.equity_curve = []
        self.signals = []
        self.daily_returns = []

        # Metrics cache
        self._metrics_cache = {}
        self._last_calculation = None

    @abstractmethod
    def run_backtest(self, *args, **kwargs) -> Tuple[Dict, pd.DataFrame]:
        """
        Run the backtest. Must be implemented by subclasses.

        Returns:
        --------
        results : Dict
            Backtest results and metrics
        trades : pd.DataFrame
            Trade history
        """
        pass

    def calculate_comprehensive_metrics(
        self,
        df: pd.DataFrame = None,
        benchmark_returns: pd.Series = None
    ) -> Dict:
        """
        Calculate comprehensive performance metrics.

        Parameters:
        -----------
        df : pd.DataFrame, optional
            Original price data for benchmark calculation
        benchmark_returns : pd.Series, optional
            Benchmark returns for comparison

        Returns:
        --------
        metrics : Dict
            Comprehensive performance metrics
        """
        if not self.trades:
            return self._empty_metrics()

        trades_df = pd.DataFrame(self.trades)
        equity_df = pd.DataFrame(self.equity_curve)

        # Basic metrics
        metrics = self._calculate_basic_metrics(trades_df, equity_df)

        # Risk metrics
        metrics.update(self._calculate_risk_metrics(equity_df))

        # Trade analysis
        metrics.update(self._calculate_trade_analysis(trades_df))

        # Time-based analysis
        metrics.update(self._calculate_time_analysis(trades_df))

        # Benchmark comparison
        if df is not None:
            metrics.update(self._calculate_benchmark_metrics(df, equity_df))

        # Advanced ratios
        metrics.update(self._calculate_advanced_ratios(equity_df))

        return metrics

    def _calculate_basic_metrics(self, trades_df: pd.DataFrame, equity_df: pd.DataFrame) -> Dict:
        """Calculate basic performance metrics."""
        final_value = equity_df['equity'].iloc[-1] if len(equity_df) > 0 else self.initial_capital
        total_return = final_value - self.initial_capital
        total_return_pct = (total_return / self.initial_capital) * 100

        return {
            'initial_capital': self.initial_capital,
            'final_value': final_value,
            'total_return': total_return,
            'total_return_pct': total_return_pct,
            'total_trades': len(trades_df),
        }

    def _calculate_risk_metrics(self, equity_df: pd.DataFrame) -> Dict:
        """Calculate risk-related metrics."""
        if len(equity_df) < 2:
            return {'sharpe_ratio': 0, 'max_drawdown': 0, 'volatility': 0}

        # Calculate returns
        equity_df = equity_df.copy()
        equity_df['returns'] = equity_df['equity'].pct_change()

        # Sharpe ratio (annualized)
        mean_return = equity_df['returns'].mean()
        std_return = equity_df['returns'].std()
        sharpe_ratio = (mean_return / std_return * np.sqrt(252)) if std_return > 0 else 0

        # Maximum drawdown
        equity_df['peak'] = equity_df['equity'].cummax()
        equity_df['drawdown'] = (equity_df['equity'] - equity_df['peak']) / equity_df['peak']
        max_drawdown = equity_df['drawdown'].min() * 100

        # Volatility (annualized)
        volatility = std_return * np.sqrt(252) * 100

        # Sortino ratio
        downside_returns = equity_df['returns'][equity_df['returns'] < 0]
        downside_std = downside_returns.std()
        sortino_ratio = (mean_return / downside_std * np.sqrt(252)) if downside_std > 0 else 0

        return {
            'sharpe_ratio': sharpe_ratio,
            'sortino_ratio': sortino_ratio,
            'max_drawdown': max_drawdown,
            'volatility': volatility,
        }

    def _calculate_trade_analysis(self, trades_df: pd.DataFrame) -> Dict:
        """Calculate trade-specific metrics."""
        if len(trades_df) == 0:
            return {}

        # Win/Loss analysis
        winning_trades = trades_df[trades_df['net_pnl'] > 0] if 'net_pnl' in trades_df.columns else trades_df[trades_df['pnl'] > 0]
        losing_trades = trades_df[trades_df['net_pnl'] <= 0] if 'net_pnl' in trades_df.columns else trades_df[trades_df['pnl'] <= 0]

        pnl_col = 'net_pnl' if 'net_pnl' in trades_df.columns else 'pnl'

        win_rate = len(winning_trades) / len(trades_df) * 100
        avg_win = winning_trades[pnl_col].mean() if len(winning_trades) > 0 else 0
        avg_loss = losing_trades[pnl_col].mean() if len(losing_trades) > 0 else 0

        # Profit factor
        gross_profit = winning_trades[pnl_col].sum() if len(winning_trades) > 0 else 0
        gross_loss = abs(losing_trades[pnl_col].sum()) if len(losing_trades) > 0 else 0
        profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')

        # Consecutive wins/losses
        trades_df['win'] = trades_df[pnl_col] > 0
        trades_df['streak'] = (trades_df['win'] != trades_df['win'].shift()).cumsum()
        streak_analysis = trades_df.groupby(['streak', 'win']).size()

        max_consecutive_wins = streak_analysis[streak_analysis.index.get_level_values(1) == True].max() if len(streak_analysis) > 0 else 0
        max_consecutive_losses = streak_analysis[streak_analysis.index.get_level_values(1) == False].max() if len(streak_analysis) > 0 else 0

        return {
            'won_trades': len(winning_trades),
            'lost_trades': len(losing_trades),
            'win_rate': win_rate,
            'avg_win': avg_win,
            'avg_loss': avg_loss,
            'best_trade': winning_trades[pnl_col].max() if len(winning_trades) > 0 else 0,
            'worst_trade': losing_trades[pnl_col].min() if len(losing_trades) > 0 else 0,
            'profit_factor': profit_factor,
            'max_consecutive_wins': max_consecutive_wins,
            'max_consecutive_losses': max_consecutive_losses,
        }

    def _calculate_time_analysis(self, trades_df: pd.DataFrame) -> Dict:
        """Calculate time-based metrics."""
        if len(trades_df) == 0 or 'entry_time' not in trades_df.columns:
            return {}

        # Holding period analysis
        if 'bars_held' in trades_df.columns:
            avg_holding_period = trades_df['bars_held'].mean()
            max_holding_period = trades_df['bars_held'].max()
            min_holding_period = trades_df['bars_held'].min()
        else:
            avg_holding_period = max_holding_period = min_holding_period = 0

        # Monthly/yearly analysis
        trades_df['entry_month'] = pd.to_datetime(trades_df['entry_time']).dt.month
        trades_df['entry_year'] = pd.to_datetime(trades_df['entry_time']).dt.year

        pnl_col = 'net_pnl' if 'net_pnl' in trades_df.columns else 'pnl'
        monthly_performance = trades_df.groupby('entry_month')[pnl_col].sum()
        best_month = monthly_performance.idxmax() if len(monthly_performance) > 0 else None
        worst_month = monthly_performance.idxmin() if len(monthly_performance) > 0 else None

        return {
            'avg_holding_period': avg_holding_period,
            'max_holding_period': max_holding_period,
            'min_holding_period': min_holding_period,
            'best_month': best_month,
            'worst_month': worst_month,
        }

    def _calculate_benchmark_metrics(self, df: pd.DataFrame, equity_df: pd.DataFrame) -> Dict:
        """Calculate benchmark comparison metrics."""
        if 'close' not in df.columns and 'Close' not in df.columns:
            return {}

        close_col = 'close' if 'close' in df.columns else 'Close'
        buy_hold_return = (df[close_col].iloc[-1] / df[close_col].iloc[0] - 1) * 100

        # Calculate correlation with benchmark
        if len(equity_df) > 1 and len(df) > 1:
            # Align data
            min_len = min(len(equity_df), len(df))
            portfolio_returns = equity_df['equity'].iloc[:min_len].pct_change().dropna()
            benchmark_returns = df[close_col].iloc[:min_len].pct_change().dropna()

            if len(portfolio_returns) > 1 and len(benchmark_returns) > 1:
                correlation = portfolio_returns.corr(benchmark_returns)
                beta = portfolio_returns.cov(benchmark_returns) / benchmark_returns.var()
            else:
                correlation = beta = 0
        else:
            correlation = beta = 0

        return {
            'buy_and_hold_return_pct': buy_hold_return,
            'correlation_with_benchmark': correlation,
            'beta': beta,
        }

    def _calculate_advanced_ratios(self, equity_df: pd.DataFrame) -> Dict:
        """Calculate advanced performance ratios."""
        if len(equity_df) < 2:
            return {}

        equity_df = equity_df.copy()
        equity_df['returns'] = equity_df['equity'].pct_change()

        # Calmar ratio (annual return / max drawdown)
        annual_return = (equity_df['equity'].iloc[-1] / equity_df['equity'].iloc[0]) ** (252 / len(equity_df)) - 1
        equity_df['peak'] = equity_df['equity'].cummax()
        equity_df['drawdown'] = (equity_df['equity'] - equity_df['peak']) / equity_df['peak']
        max_dd = abs(equity_df['drawdown'].min())
        calmar_ratio = annual_return / max_dd if max_dd > 0 else 0

        # Information ratio (assuming benchmark return is 0)
        mean_excess_return = equity_df['returns'].mean()
        tracking_error = equity_df['returns'].std()
        information_ratio = mean_excess_return / tracking_error if tracking_error > 0 else 0

        return {
            'calmar_ratio': calmar_ratio,
            'information_ratio': information_ratio,
            'annual_return': annual_return * 100,
        }

    def _empty_metrics(self) -> Dict:
        """Return empty metrics when no trades exist."""
        return {
            'initial_capital': self.initial_capital,
            'final_value': self.initial_capital,
            'total_return': 0,
            'total_return_pct': 0,
            'total_trades': 0,
            'won_trades': 0,
            'lost_trades': 0,
            'win_rate': 0,
            'sharpe_ratio': 0,
            'max_drawdown': 0,
        }

    def generate_detailed_report(self, results: Dict, save_path: Optional[str] = None) -> str:
        """
        Generate a comprehensive text report.

        Parameters:
        -----------
        results : Dict
            Backtest results
        save_path : str, optional
            Path to save the report

        Returns:
        --------
        report : str
            Formatted report text
        """
        report_lines = []
        report_lines.append("=" * 100)
        report_lines.append("COMPREHENSIVE BACKTEST REPORT")
        report_lines.append("=" * 100)
        report_lines.append(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report_lines.append("")

        # Capital Summary
        report_lines.append("CAPITAL SUMMARY")
        report_lines.append("-" * 50)
        report_lines.append(f"Initial Capital:           ${results.get('initial_capital', 0):,.2f}")
        report_lines.append(f"Final Value:               ${results.get('final_value', 0):,.2f}")
        report_lines.append(f"Total Return:              ${results.get('total_return', 0):,.2f}")
        report_lines.append(f"Total Return %:            {results.get('total_return_pct', 0):.2f}%")
        if 'buy_and_hold_return_pct' in results:
            report_lines.append(f"Buy & Hold Return %:       {results['buy_and_hold_return_pct']:.2f}%")
        if 'annual_return' in results:
            report_lines.append(f"Annualized Return %:       {results['annual_return']:.2f}%")
        report_lines.append("")

        # Trade Statistics
        report_lines.append("TRADE STATISTICS")
        report_lines.append("-" * 50)
        report_lines.append(f"Total Trades:              {results.get('total_trades', 0)}")
        report_lines.append(f"Winning Trades:            {results.get('won_trades', 0)}")
        report_lines.append(f"Losing Trades:             {results.get('lost_trades', 0)}")
        report_lines.append(f"Win Rate:                  {results.get('win_rate', 0):.2f}%")
        report_lines.append(f"Average Win:               ${results.get('avg_win', 0):.2f}")
        report_lines.append(f"Average Loss:              ${results.get('avg_loss', 0):.2f}")
        report_lines.append(f"Best Trade:                ${results.get('best_trade', 0):.2f}")
        report_lines.append(f"Worst Trade:               ${results.get('worst_trade', 0):.2f}")
        report_lines.append(f"Profit Factor:             {results.get('profit_factor', 0):.2f}")
        if 'max_consecutive_wins' in results:
            report_lines.append(f"Max Consecutive Wins:      {results['max_consecutive_wins']}")
        if 'max_consecutive_losses' in results:
            report_lines.append(f"Max Consecutive Losses:    {results['max_consecutive_losses']}")
        report_lines.append("")

        # Risk Metrics
        report_lines.append("RISK METRICS")
        report_lines.append("-" * 50)
        report_lines.append(f"Maximum Drawdown:          {results.get('max_drawdown', 0):.2f}%")
        report_lines.append(f"Volatility (Annual):       {results.get('volatility', 0):.2f}%")
        report_lines.append(f"Sharpe Ratio:              {results.get('sharpe_ratio', 0):.2f}")
        if 'sortino_ratio' in results:
            report_lines.append(f"Sortino Ratio:             {results['sortino_ratio']:.2f}")
        if 'calmar_ratio' in results:
            report_lines.append(f"Calmar Ratio:              {results['calmar_ratio']:.2f}")
        if 'information_ratio' in results:
            report_lines.append(f"Information Ratio:         {results['information_ratio']:.2f}")
        report_lines.append("")

        # Time Analysis
        if 'avg_holding_period' in results:
            report_lines.append("TIME ANALYSIS")
            report_lines.append("-" * 50)
            report_lines.append(f"Avg Holding Period:        {results['avg_holding_period']:.1f} bars")
            report_lines.append(f"Max Holding Period:        {results.get('max_holding_period', 0):.0f} bars")
            report_lines.append(f"Min Holding Period:        {results.get('min_holding_period', 0):.0f} bars")
            if results.get('best_month'):
                report_lines.append(f"Best Month:                {results['best_month']}")
            if results.get('worst_month'):
                report_lines.append(f"Worst Month:               {results['worst_month']}")
            report_lines.append("")

        # Benchmark Comparison
        if 'correlation_with_benchmark' in results:
            report_lines.append("BENCHMARK COMPARISON")
            report_lines.append("-" * 50)
            report_lines.append(f"Correlation:               {results['correlation_with_benchmark']:.2f}")
            report_lines.append(f"Beta:                      {results.get('beta', 0):.2f}")
            report_lines.append("")

        report_lines.append("=" * 100)

        report_text = "\n".join(report_lines)

        if save_path:
            with open(save_path, 'w') as f:
                f.write(report_text)
            print(f"Report saved to: {save_path}")

        return report_text

    def create_comprehensive_visualizations(
        self,
        results: Dict,
        df: pd.DataFrame = None,
        save_dir: Optional[str] = None,
        show_plots: bool = True,
        model_name: Optional[str] = None,
        path_suffix: Optional[str] = None
    ) -> Dict[str, str]:
        """
        Create comprehensive visualization suite with separate interactive HTML files.

        Creates 5 separate Plotly HTML files:
        1. Performance Overview (equity, drawdown, returns, metrics)
        2. Trade Analysis (P&L over time, win/loss, duration, monthly)
        3. Risk Analysis (rolling Sharpe, volatility, underwater, risk-return)
        4. Monthly Heatmap
        5. Trade Distribution (P&L dist, box plot, CDF, Q-Q plot)

        Parameters:
        -----------
        results : Dict
            Backtest results
        df : pd.DataFrame, optional
            Original price data
        save_dir : str, optional
            Directory to save plots
        show_plots : bool
            Whether to display plots
        model_name : str, optional
            Model name for titles

        Returns:
        --------
        saved_files : Dict[str, str]
            Dictionary of plot names and their file paths
        """
        try:
            import plotly.graph_objects as go
            from plotly.subplots import make_subplots
            import plotly.express as px
            import webbrowser
            import os
        except ImportError:
            print("⚠ Plotly not installed. Falling back to matplotlib.")
            return self._create_comprehensive_visualizations_matplotlib(results,
                                        df, save_dir, show_plots, path_suffix)

        saved_files = {}

        if save_dir:
            if path_suffix:
                save_dir = os.path.join(save_dir, path_suffix)
            Path(save_dir).mkdir(parents=True, exist_ok=True)
        else:
            save_dir = 'backtest_results'
            if path_suffix:
                save_dir = os.path.join(save_dir, path_suffix)
            Path(save_dir).mkdir(parents=True, exist_ok=True)

        print("\n" + "="*70)
        print("CREATING COMPREHENSIVE PLOTLY VISUALIZATIONS")
        print("="*70)

        # Extract data
        equity_curve = results.get('equity_curve', self.equity_curve if hasattr(self, 'equity_curve') else None)
        trades = results.get('trades', self.trades if hasattr(self, 'trades') else None)

        if equity_curve is None or len(equity_curve) == 0:
            print("  ⚠ No equity curve data available")
            return saved_files

        equity_df = pd.DataFrame(equity_curve)

        # Create comprehensive figure with all plots stacked vertically
        # Total: 17 plots in 5 rows
        fig = make_subplots(
            rows=9, cols=2,
            row_heights=[0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.08, 0.08],
            subplot_titles=(
                # Row 1: Performance Overview
                'Equity Curve', 'Drawdown (Underwater)',
                # Row 2: Performance Overview continued
                'Returns Distribution', 'Performance Metrics',
                # Row 3: Trade Analysis
                'Cumulative P&L by Trade', 'Win/Loss Distribution',
                # Row 4: Trade Analysis continued
                'Trade Duration Analysis', 'Monthly Performance',
                # Row 5: Risk Analysis
                'Rolling Sharpe Ratio', 'Rolling Volatility',
                # Row 6: Risk Analysis continued
                'Underwater Curve', 'Risk-Return Scatter',
                # Row 7: Trade Distribution
                'P&L Distribution', 'Win/Loss Box Plot',
                # Row 8: Trade Distribution continued
                'Cumulative Distribution', 'Q-Q Plot (Normal)',
                # Row 9: Monthly Heatmap (spans 2 columns)
                'Monthly Returns Heatmap', None
            ),
            specs=[
                # Row 1
                [{'type': 'scatter'}, {'type': 'scatter'}],
                # Row 2
                [{'type': 'histogram'}, {'type': 'table'}],
                # Row 3
                [{'type': 'scatter'}, {'type': 'histogram'}],
                # Row 4
                [{'type': 'box'}, {'type': 'bar'}],
                # Row 5
                [{'type': 'scatter'}, {'type': 'scatter'}],
                # Row 6
                [{'type': 'scatter'}, {'type': 'scatter'}],
                # Row 7
                [{'type': 'histogram'}, {'type': 'box'}],
                # Row 8
                [{'type': 'scatter'}, {'type': 'scatter'}],
                # Row 9
                [{'type': 'heatmap', 'colspan': 2}, None]
            ],
            vertical_spacing=0.04,
            horizontal_spacing=0.08
        )

        # ===== ROW 1: PERFORMANCE OVERVIEW =====
        # 1. Equity Curve
        fig.add_trace(
            go.Scatter(
                x=equity_df['timestamp'],
                y=equity_df['equity'],
                mode='lines',
                name='Equity',
                line=dict(color='blue', width=2),
                fill='tozeroy',
                fillcolor='rgba(0,100,255,0.2)',
                showlegend=False
            ),
            row=1, col=1
        )
        initial_capital = results.get('initial_capital', 10000)
        # Add initial capital line
        fig.add_trace(
            go.Scatter(
                x=[equity_df['timestamp'].iloc[0], equity_df['timestamp'].iloc[-1]],
                y=[initial_capital, initial_capital],
                mode='lines',
                line=dict(color='gray', dash='dash', width=1),
                name='Initial Capital',
                showlegend=False,
                hoverinfo='skip'
            ),
            row=1, col=1
        )

        # 2. Drawdown
        peak = equity_df['equity'].expanding().max()
        drawdown = ((equity_df['equity'] - peak) / peak * 100)
        fig.add_trace(
            go.Scatter(
                x=equity_df['timestamp'],
                y=drawdown,
                mode='lines',
                name='Drawdown',
                line=dict(color='red', width=2),
                fill='tozeroy',
                fillcolor='rgba(255,0,0,0.2)',
                showlegend=False
            ),
            row=1, col=2
        )

        # ===== ROW 2: PERFORMANCE OVERVIEW CONTINUED =====
        # 3. Returns Distribution
        returns = equity_df['equity'].pct_change().dropna() * 100
        fig.add_trace(
            go.Histogram(
                x=returns,
                name='Returns',
                marker_color='skyblue',
                nbinsx=50,
                showlegend=False
            ),
            row=2, col=1
        )
        mean_return = returns.mean()
        # Add mean return line
        fig.add_trace(
            go.Scatter(
                x=[mean_return, mean_return],
                y=[0, len(returns) * 0.1],  # Approximate height for histogram
                mode='lines',
                line=dict(color='red', dash='dash', width=2),
                name=f'Mean: {mean_return:.2f}%',
                showlegend=False,
                hoverinfo='skip'
            ),
            row=2, col=1
        )

        # 4. Performance Metrics Table
        metrics_data = [
            ['Total Return', f"{results.get('total_return_pct', 0):.2f}%"],
            ['Sharpe Ratio', f"{results.get('sharpe_ratio', 0):.2f}"],
            ['Max Drawdown', f"{results.get('max_drawdown', 0):.2f}%"],
            ['Win Rate', f"{results.get('win_rate', 0):.1f}%"],
            ['Total Trades', f"{results.get('total_trades', 0)}"],
            ['Profit Factor', f"{results.get('profit_factor', 0):.2f}"],
            ['Avg Win', f"${results.get('avg_win', 0):.2f}"],
            ['Avg Loss', f"${results.get('avg_loss', 0):.2f}"]
        ]
        fig.add_trace(
            go.Table(
                header=dict(
                    values=['<b>Metric</b>', '<b>Value</b>'],
                    fill_color='paleturquoise',
                    align='left',
                    font=dict(size=11)
                ),
                cells=dict(
                    values=[[row[0] for row in metrics_data], [row[1] for row in metrics_data]],
                    fill_color='lavender',
                    align='left',
                    font=dict(size=10)
                )
            ),
            row=2, col=2
        )

        # ===== ROW 3-4: TRADE ANALYSIS =====
        if trades is not None and len(trades) > 0:
            trades_df = pd.DataFrame(trades)

            # 5. Cumulative P&L
            cumulative_pnl = trades_df['net_pnl'].cumsum()
            fig.add_trace(
                go.Scatter(
                    x=list(range(len(cumulative_pnl))),
                    y=cumulative_pnl,
                    mode='lines+markers',
                    name='Cumulative P&L',
                    line=dict(color='blue', width=2),
                    marker=dict(size=3),
                    showlegend=False
                ),
                row=3, col=1
            )

            # 6. Win/Loss Distribution
            wins = trades_df[trades_df['net_pnl'] > 0]['net_pnl']
            losses = trades_df[trades_df['net_pnl'] <= 0]['net_pnl']
            fig.add_trace(
                go.Histogram(x=wins, name='Wins', marker_color='green', opacity=0.7, nbinsx=20),
                row=3, col=2
            )
            fig.add_trace(
                go.Histogram(x=losses, name='Losses', marker_color='red', opacity=0.7, nbinsx=20),
                row=3, col=2
            )

            # 7. Trade Duration Box Plot
            if 'duration_bars' not in trades_df.columns:
                if 'entry_time' in trades_df.columns and 'exit_time' in trades_df.columns:
                    trades_df['entry_time'] = pd.to_datetime(trades_df['entry_time'])
                    trades_df['exit_time'] = pd.to_datetime(trades_df['exit_time'])
                    trades_df['duration_bars'] = (trades_df['exit_time'] - trades_df['entry_time']).dt.total_seconds() / 3600
                elif 'bars_held' in trades_df.columns:
                    trades_df['duration_bars'] = trades_df['bars_held']

            if 'duration_bars' in trades_df.columns and len(trades_df['duration_bars'].dropna()) > 0:
                fig.add_trace(
                    go.Box(
                        y=trades_df['duration_bars'],
                        name='Duration',
                        marker_color='orange',
                        boxmean='sd',
                        showlegend=False
                    ),
                    row=4, col=1
                )

            # 8. Monthly Performance
            if 'exit_time' in trades_df.columns:
                trades_df['exit_time'] = pd.to_datetime(trades_df['exit_time'])
                trades_df['month'] = trades_df['exit_time'].dt.month
                monthly_pnl = trades_df.groupby('month')['net_pnl'].sum()
                colors = ['green' if x > 0 else 'red' for x in monthly_pnl.values]
                fig.add_trace(
                    go.Bar(
                        x=monthly_pnl.index,
                        y=monthly_pnl.values,
                        marker_color=colors,
                        name='Monthly P&L',
                        text=[f'${x:.0f}' for x in monthly_pnl.values],
                        textposition='auto',
                        showlegend=False
                    ),
                    row=4, col=2
                )

        # ===== ROW 5-6: RISK ANALYSIS =====
        equity_df['returns'] = equity_df['equity'].pct_change()
        window = min(30, len(equity_df) // 4)

        # 9. Rolling Sharpe
        if window > 1:
            rolling_sharpe = (equity_df['returns'].rolling(window).mean() /
                            equity_df['returns'].rolling(window).std() * np.sqrt(252))
            fig.add_trace(
                go.Scatter(
                    x=equity_df['timestamp'],
                    y=rolling_sharpe,
                    mode='lines',
                    name=f'Sharpe ({window}d)',
                    line=dict(color='purple', width=2),
                    showlegend=False
                ),
                row=5, col=1
            )

        # 10. Rolling Volatility
        if window > 1:
            rolling_vol = equity_df['returns'].rolling(window).std() * np.sqrt(252) * 100
            fig.add_trace(
                go.Scatter(
                    x=equity_df['timestamp'],
                    y=rolling_vol,
                    mode='lines',
                    name=f'Volatility ({window}d)',
                    line=dict(color='orange', width=2),
                    showlegend=False
                ),
                row=5, col=2
            )

        # 11. Underwater Curve
        fig.add_trace(
            go.Scatter(
                x=equity_df['timestamp'],
                y=drawdown,
                mode='lines',
                name='Drawdown',
                line=dict(color='red', width=2),
                fill='tozeroy',
                fillcolor='rgba(255,0,0,0.2)',
                showlegend=False
            ),
            row=6, col=1
        )

        # 12. Risk-Return Scatter
        if trades is not None and len(trades) > 0 and 'duration_bars' in trades_df.columns:
            if len(trades_df['duration_bars'].dropna()) > 0:
                fig.add_trace(
                    go.Scatter(
                        x=trades_df['duration_bars'],
                        y=trades_df['net_pnl'],
                        mode='markers',
                        name='Trades',
                        marker=dict(
                            color=trades_df['net_pnl'],
                            colorscale='RdYlGn',
                            size=6,
                            opacity=0.6,
                            showscale=True,
                            colorbar=dict(title='P&L', len=0.3, y=0.25)
                        ),
                        showlegend=False
                    ),
                    row=6, col=2
                )

        # ===== ROW 7-8: TRADE DISTRIBUTION =====
        if trades is not None and len(trades) > 0:
            pnl_col = 'net_pnl'

            # 13. P&L Distribution
            fig.add_trace(
                go.Histogram(
                    x=trades_df[pnl_col],
                    name='P&L',
                    marker_color='skyblue',
                    nbinsx=30,
                    showlegend=False
                ),
                row=7, col=1
            )
            mean_pnl = trades_df[pnl_col].mean()
            # Add mean line as a vertical line trace
            fig.add_trace(
                go.Scatter(
                    x=[mean_pnl, mean_pnl],
                    y=[0, trades_df[pnl_col].value_counts().max()],
                    mode='lines',
                    line=dict(color='red', dash='dash', width=2),
                    showlegend=False,
                    hoverinfo='skip'
                ),
                row=7, col=1
            )
            # Add zero line
            fig.add_trace(
                go.Scatter(
                    x=[0, 0],
                    y=[0, trades_df[pnl_col].value_counts().max()],
                    mode='lines',
                    line=dict(color='black', dash='dash', width=1),
                    showlegend=False,
                    hoverinfo='skip'
                ),
                row=7, col=1
            )

            # 14. Win/Loss Box Plot
            fig.add_trace(
                go.Box(y=wins, name='Wins', marker_color='green', boxmean='sd'),
                row=7, col=2
            )
            fig.add_trace(
                go.Box(y=losses, name='Losses', marker_color='red', boxmean='sd'),
                row=7, col=2
            )

            # 15. Cumulative Distribution
            sorted_pnl = np.sort(trades_df[pnl_col])
            cumulative_prob = np.arange(1, len(sorted_pnl) + 1) / len(sorted_pnl)
            fig.add_trace(
                go.Scatter(
                    x=sorted_pnl,
                    y=cumulative_prob,
                    mode='lines+markers',
                    name='CDF',
                    line=dict(color='blue', width=2),
                    marker=dict(size=2),
                    showlegend=False
                ),
                row=8, col=1
            )

            # 16. Q-Q Plot
            try:
                from scipy import stats
                theoretical_quantiles, sample_quantiles = stats.probplot(trades_df[pnl_col], dist="norm")
                fig.add_trace(
                    go.Scatter(
                        x=theoretical_quantiles[0],
                        y=theoretical_quantiles[1],
                        mode='markers',
                        name='Q-Q',
                        marker=dict(color='blue', size=4),
                        showlegend=False
                    ),
                    row=8, col=2
                )
                min_val = min(theoretical_quantiles[0].min(), theoretical_quantiles[1].min())
                max_val = max(theoretical_quantiles[0].max(), theoretical_quantiles[1].max())
                fig.add_trace(
                    go.Scatter(
                        x=[min_val, max_val],
                        y=[min_val, max_val],
                        mode='lines',
                        name='Normal',
                        line=dict(color='red', dash='dash'),
                        showlegend=False
                    ),
                    row=8, col=2
                )
            except:
                pass

            # 17. Monthly Heatmap
            if 'exit_time' in trades_df.columns:
                trades_df['year'] = trades_df['exit_time'].dt.year
                monthly_returns = trades_df.groupby(['year', 'month'])['net_pnl'].sum().reset_index()
                heatmap_data = monthly_returns.pivot(index='year', columns='month', values='net_pnl')
                heatmap_data = heatmap_data.fillna(0)

                fig.add_trace(
                    go.Heatmap(
                        z=heatmap_data.values,
                        x=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'][:len(heatmap_data.columns)],
                        y=heatmap_data.index,
                        colorscale='RdYlGn',
                        zmid=0,
                        text=heatmap_data.values,
                        texttemplate='$%{text:.0f}',
                        textfont={"size": 9},
                        colorbar=dict(title='P&L', len=0.3, y=0.05),
                        showscale=True
                    ),
                    row=9, col=1
                )

        # Update layout
        model_str = f" - {model_name}" if model_name else ""
        fig.update_layout(
            title=dict(
                text=f'Comprehensive Backtest Analysis{model_str}<br><sub>Total Return: {results.get("total_return_pct", 0):.2f}% | Win Rate: {results.get("win_rate", 0):.1f}% | Sharpe: {results.get("sharpe_ratio", 0):.2f}</sub>',
                x=0.5,
                xanchor='center',
                font=dict(size=16)
            ),
            height=3000,  # Tall layout for vertical stacking
            showlegend=True,
            template='plotly_white',
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )

        # Update axes labels (skip row 2 col 2 which is a table)
        fig.update_xaxes(title_text="Date", row=1, col=1)
        fig.update_yaxes(title_text="Equity ($)", row=1, col=1)
        fig.update_xaxes(title_text="Date", row=1, col=2)
        fig.update_yaxes(title_text="Drawdown (%)", row=1, col=2)
        fig.update_xaxes(title_text="Return (%)", row=2, col=1)
        fig.update_yaxes(title_text="Frequency", row=2, col=1)
        # Row 2, Col 2 is a table - no axes to update
        fig.update_xaxes(title_text="Trade #", row=3, col=1)
        fig.update_yaxes(title_text="Cumulative P&L ($)", row=3, col=1)
        fig.update_xaxes(title_text="P&L ($)", row=3, col=2)
        fig.update_yaxes(title_text="Frequency", row=3, col=2)
        fig.update_yaxes(title_text="Duration (bars)", row=4, col=1)
        fig.update_xaxes(title_text="Month", row=4, col=2)
        fig.update_yaxes(title_text="P&L ($)", row=4, col=2)
        fig.update_xaxes(title_text="Date", row=5, col=1)
        fig.update_yaxes(title_text="Sharpe Ratio", row=5, col=1)
        fig.update_xaxes(title_text="Date", row=5, col=2)
        fig.update_yaxes(title_text="Volatility (%)", row=5, col=2)
        fig.update_xaxes(title_text="Date", row=6, col=1)
        fig.update_yaxes(title_text="Drawdown (%)", row=6, col=1)
        fig.update_xaxes(title_text="Holding Period (bars)", row=6, col=2)
        fig.update_yaxes(title_text="P&L ($)", row=6, col=2)
        fig.update_xaxes(title_text="P&L ($)", row=7, col=1)
        fig.update_yaxes(title_text="Frequency", row=7, col=1)
        fig.update_yaxes(title_text="P&L ($)", row=7, col=2)
        fig.update_xaxes(title_text="P&L ($)", row=8, col=1)
        fig.update_yaxes(title_text="Cumulative Probability", row=8, col=1)
        fig.update_xaxes(title_text="Theoretical Quantiles", row=8, col=2)
        fig.update_yaxes(title_text="Sample Quantiles", row=8, col=2)
        fig.update_xaxes(title_text="Month", row=9, col=1)
        fig.update_yaxes(title_text="Year", row=9, col=1)

        # Save HTML
        timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
        model_suffix = f"_{model_name}" if model_name else ""
        filepath = Path(save_dir) / f'comprehensive_analysis{model_suffix}_{timestamp}'
        fig.write_html(str(filepath)+'.html')
        fig.write_image(str(filepath)+'.png')
        saved_files['comprehensive'] = str(filepath)

        print(f"  ✓ Comprehensive analysis saved to: {filepath.name}")
        print(f"  ✓ Total plots: 17 (all in one HTML file)")
        print("="*70)
        print(f"✓ All visualizations saved to: {save_dir}")
        print("="*70)

        # Open in browser
        if show_plots:
            webbrowser.open('file://' + os.path.abspath(str(filepath)))

        return saved_files

        # Extract data from results or self
        equity_curve = results.get('equity_curve', self.equity_curve if hasattr(self, 'equity_curve') else None)
        trades = results.get('trades', self.trades if hasattr(self, 'trades') else None)

        # Debug output
        equity_available = False
        trades_available = False

        if equity_curve is not None:
            try:
                equity_len = len(equity_curve)
                equity_available = equity_len > 0
                if equity_available:
                    print(f"  Debug - Equity curve points: {equity_len}")
            except:
                equity_available = False

        if trades is not None:
            try:
                trades_len = len(trades)
                trades_available = trades_len > 0
                if trades_available:
                    print(f"  Debug - Number of trades: {trades_len}")
                    # Show available columns
                    trades_df_temp = pd.DataFrame(trades)
                    print(f"  Debug - Trades columns: {list(trades_df_temp.columns)}")
            except:
                trades_available = False

        print(f"  Debug - Equity curve available: {equity_available}")
        print(f"  Debug - Trades available: {trades_available}")

        # 1. Equity Curve
        if equity_curve is not None and equity_available:
            equity_df = pd.DataFrame(equity_curve)
            fig.add_trace(
                go.Scatter(
                    x=equity_df['timestamp'],
                    y=equity_df['equity'],
                    mode='lines',
                    name='Equity',
                    line=dict(color='blue', width=2),
                    fill='tozeroy',
                    fillcolor='rgba(0,100,255,0.2)'
                ),
                row=1, col=1
            )

            # Add initial capital line
            initial_capital = results.get('initial_capital', 10000)
            fig.add_hline(
                y=initial_capital,
                line_dash="dash",
                line_color="gray",
                annotation_text="Initial Capital",
                row=1, col=1
            )

        # 2. Drawdown (Underwater)
        if equity_curve is not None and equity_available:
            equity_df = pd.DataFrame(equity_curve)
            peak = equity_df['equity'].expanding().max()
            drawdown = ((equity_df['equity'] - peak) / peak * 100)

            fig.add_trace(
                go.Scatter(
                    x=equity_df['timestamp'],
                    y=drawdown,
                    mode='lines',
                    name='Drawdown',
                    line=dict(color='red', width=2),
                    fill='tozeroy',
                    fillcolor='rgba(255,0,0,0.2)'
                ),
                row=1, col=2
            )

        # 3. Daily Returns Distribution
        if equity_curve is not None and equity_available:
            equity_df = pd.DataFrame(equity_curve)
            returns = equity_df['equity'].pct_change().dropna() * 100

            fig.add_trace(
                go.Histogram(
                    x=returns,
                    name='Returns',
                    marker_color='skyblue',
                    nbinsx=50
                ),
                row=1, col=3
            )

            # Add mean line
            mean_return = returns.mean()
            fig.add_vline(
                x=mean_return,
                line_dash="dash",
                line_color="red",
                annotation_text=f"Mean: {mean_return:.2f}%",
                row=1, col=3
            )

        # 4. Cumulative P&L by Trade
        if trades is not None and trades_available:
            trades_df = pd.DataFrame(trades)
            cumulative_pnl = trades_df['net_pnl'].cumsum()

            fig.add_trace(
                go.Scatter(
                    x=list(range(len(cumulative_pnl))),
                    y=cumulative_pnl,
                    mode='lines+markers',
                    name='Cumulative P&L',
                    line=dict(color='blue', width=2),
                    marker=dict(size=4)
                ),
                row=2, col=1
            )

        # 5. Win/Loss Distribution (Histogram)
        if trades is not None and trades_available:
            trades_df = pd.DataFrame(trades)
            wins = trades_df[trades_df['net_pnl'] > 0]['net_pnl']
            losses = trades_df[trades_df['net_pnl'] <= 0]['net_pnl']

            fig.add_trace(
                go.Histogram(
                    x=wins,
                    name='Wins',
                    marker_color='green',
                    opacity=0.7,
                    nbinsx=20
                ),
                row=2, col=2
            )

            fig.add_trace(
                go.Histogram(
                    x=losses,
                    name='Losses',
                    marker_color='red',
                    opacity=0.7,
                    nbinsx=20
                ),
                row=2, col=2
            )

        # 6. P&L vs Holding Period (Scatter)
        if trades is not None and trades_available:
            trades_df = pd.DataFrame(trades)

            # Calculate duration_bars if not present
            if 'duration_bars' not in trades_df.columns:
                if 'entry_time' in trades_df.columns and 'exit_time' in trades_df.columns:
                    trades_df['entry_time'] = pd.to_datetime(trades_df['entry_time'])
                    trades_df['exit_time'] = pd.to_datetime(trades_df['exit_time'])
                    # Estimate bars (assuming 1 bar = 1 hour for hourly data)
                    trades_df['duration_bars'] = (trades_df['exit_time'] - trades_df['entry_time']).dt.total_seconds() / 3600
                elif 'bars_held' in trades_df.columns:
                    trades_df['duration_bars'] = trades_df['bars_held']

            if 'duration_bars' in trades_df.columns and len(trades_df['duration_bars'].dropna()) > 0:
                fig.add_trace(
                    go.Scatter(
                        x=trades_df['duration_bars'],
                        y=trades_df['net_pnl'],
                        mode='markers',
                        name='Trades',
                        marker=dict(
                            color=trades_df['net_pnl'],
                            colorscale='RdYlGn',
                            size=8,
                            opacity=0.6,
                            colorbar=dict(title='P&L')
                        )
                    ),
                    row=2, col=3
                )
            else:
                print(f"  ⚠ Skipping P&L vs Holding Period plot - duration data not available")

        # 7. Monthly P&L (Bar Chart)
        if trades is not None and trades_available:
            trades_df = pd.DataFrame(trades)
            if 'exit_time' in trades_df.columns:
                trades_df['exit_time'] = pd.to_datetime(trades_df['exit_time'])
                trades_df['month'] = trades_df['exit_time'].dt.month
                monthly_pnl = trades_df.groupby('month')['net_pnl'].sum()

                colors = ['green' if x > 0 else 'red' for x in monthly_pnl.values]
                fig.add_trace(
                    go.Bar(
                        x=monthly_pnl.index,
                        y=monthly_pnl.values,
                        marker_color=colors,
                        name='Monthly P&L',
                        text=[f'${x:.0f}' for x in monthly_pnl.values],
                        textposition='auto'
                    ),
                    row=3, col=1
                )

        # 8. Trade Duration Box Plot
        if trades is not None and trades_available:
            trades_df = pd.DataFrame(trades)

            # Calculate duration_bars if not present (same logic as above)
            if 'duration_bars' not in trades_df.columns:
                if 'entry_time' in trades_df.columns and 'exit_time' in trades_df.columns:
                    trades_df['entry_time'] = pd.to_datetime(trades_df['entry_time'])
                    trades_df['exit_time'] = pd.to_datetime(trades_df['exit_time'])
                    trades_df['duration_bars'] = (trades_df['exit_time'] - trades_df['entry_time']).dt.total_seconds() / 3600
                elif 'bars_held' in trades_df.columns:
                    trades_df['duration_bars'] = trades_df['bars_held']

            if 'duration_bars' in trades_df.columns and len(trades_df['duration_bars'].dropna()) > 0:
                fig.add_trace(
                    go.Box(
                        y=trades_df['duration_bars'],
                        name='Duration',
                        marker_color='orange',
                        boxmean='sd'
                    ),
                    row=3, col=2
                )
            else:
                print(f"  ⚠ Skipping Trade Duration Box Plot - duration data not available")

        # 9. Rolling Sharpe Ratio
        if equity_curve is not None and equity_available:
            equity_df = pd.DataFrame(equity_curve)
            equity_df['returns'] = equity_df['equity'].pct_change()
            window = min(30, len(equity_df) // 4)

            if window > 1:
                rolling_sharpe = (equity_df['returns'].rolling(window).mean() /
                                equity_df['returns'].rolling(window).std() * np.sqrt(252))

                fig.add_trace(
                    go.Scatter(
                        x=equity_df['timestamp'],
                        y=rolling_sharpe,
                        mode='lines',
                        name=f'Sharpe ({window}d)',
                        line=dict(color='purple', width=2)
                    ),
                    row=3, col=3
                )

        # 10. Rolling Volatility
        if equity_curve is not None and equity_available:
            equity_df = pd.DataFrame(equity_curve)
            if 'returns' not in equity_df.columns:
                equity_df['returns'] = equity_df['equity'].pct_change()
            window = min(30, len(equity_df) // 4)

            if window > 1:
                rolling_vol = equity_df['returns'].rolling(window).std() * np.sqrt(252) * 100

                fig.add_trace(
                    go.Scatter(
                        x=equity_df['timestamp'],
                        y=rolling_vol,
                        mode='lines',
                        name=f'Volatility ({window}d)',
                        line=dict(color='orange', width=2)
                    ),
                    row=4, col=1
                )

        # 11. Trade P&L Distribution (Overall)
        if trades is not None and trades_available:
            trades_df = pd.DataFrame(trades)
            fig.add_trace(
                go.Histogram(
                    x=trades_df['net_pnl'],
                    name='P&L Distribution',
                    marker_color='steelblue',
                    nbinsx=30
                ),
                row=4, col=2
            )

            # Add vertical line at zero
            fig.add_vline(
                x=0,
                line_dash="dash",
                line_color="black",
                row=4, col=2
            )

        # 12. Monthly Returns Heatmap
        if trades is not None and trades_available:
            trades_df = pd.DataFrame(trades)
            if 'exit_time' in trades_df.columns:
                trades_df['exit_time'] = pd.to_datetime(trades_df['exit_time'])
                trades_df['year'] = trades_df['exit_time'].dt.year
                trades_df['month'] = trades_df['exit_time'].dt.month

                # Group by year and month
                monthly_returns = trades_df.groupby(['year', 'month'])['net_pnl'].sum().reset_index()

                # Pivot for heatmap
                heatmap_data = monthly_returns.pivot(index='year', columns='month', values='net_pnl')
                heatmap_data = heatmap_data.fillna(0)

                fig.add_trace(
                    go.Heatmap(
                        z=heatmap_data.values,
                        x=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'][:len(heatmap_data.columns)],
                        y=heatmap_data.index,
                        colorscale='RdYlGn',
                        zmid=0,
                        name='Monthly Returns',
                        colorbar=dict(title='P&L')
                    ),
                    row=4, col=3
                )

        # Update layout
        fig.update_layout(
            title=dict(
                text=f'Comprehensive Backtest Analysis ({model_name})<br><sub>Total Return: {results.get("total_return_pct", 0):.2f}% | Win Rate: {results.get("win_rate", 0):.1f}% | Sharpe: {results.get("sharpe_ratio", 0):.2f}</sub>',
                x=0.5,
                xanchor='center'
            ),
            height=1600,
            showlegend=True,
            template='plotly_white'
        )

        # Update axes labels for all subplots
        # Row 1
        fig.update_xaxes(title_text="Date", row=1, col=1)
        fig.update_yaxes(title_text="Equity ($)", row=1, col=1)
        fig.update_xaxes(title_text="Date", row=1, col=2)
        fig.update_yaxes(title_text="Drawdown (%)", row=1, col=2)
        fig.update_xaxes(title_text="Daily Return (%)", row=1, col=3)
        fig.update_yaxes(title_text="Frequency", row=1, col=3)

        # Row 2
        fig.update_xaxes(title_text="Trade Number", row=2, col=1)
        fig.update_yaxes(title_text="Cumulative P&L ($)", row=2, col=1)
        fig.update_xaxes(title_text="P&L ($)", row=2, col=2)
        fig.update_yaxes(title_text="Frequency", row=2, col=2)
        fig.update_xaxes(title_text="Holding Period (bars)", row=2, col=3)
        fig.update_yaxes(title_text="P&L ($)", row=2, col=3)

        # Row 3
        fig.update_xaxes(title_text="Month", row=3, col=1)
        fig.update_yaxes(title_text="P&L ($)", row=3, col=1)
        fig.update_yaxes(title_text="Duration (bars)", row=3, col=2)
        fig.update_xaxes(title_text="Date", row=3, col=3)
        fig.update_yaxes(title_text="Sharpe Ratio", row=3, col=3)

        # Row 4
        fig.update_xaxes(title_text="Date", row=4, col=1)
        fig.update_yaxes(title_text="Volatility (%)", row=4, col=1)
        fig.update_xaxes(title_text="P&L ($)", row=4, col=2)
        fig.update_yaxes(title_text="Frequency", row=4, col=2)
        fig.update_xaxes(title_text="Month", row=4, col=3)
        fig.update_yaxes(title_text="Year", row=4, col=3)

        # Save HTML
        timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
        model_name = results.get('model_name', 'backtest')
        filepath = Path(save_dir) / f'comprehensive_analysis_{model_name}_{timestamp}.html'
        fig.write_html(str(filepath))
        saved_files['comprehensive'] = str(filepath)

        print(f"  ✓ Comprehensive analysis saved to: {filepath}")

        # Open in browser
        if show_plots:
            webbrowser.open('file://' + os.path.abspath(str(filepath)))

        return saved_files

    def _create_comprehensive_visualizations_matplotlib(
        self,
        results: Dict,
        df: pd.DataFrame = None,
        save_dir: Optional[str] = None,
        show_plots: bool = True,
        path_suffix: str = None
    ) -> Dict[str, str]:
        """
        Fallback matplotlib version of comprehensive visualizations.
        """
        saved_files = {}

        if save_dir:
            if path_suffix:
                save_dir = os.path.join(save_dir, path_suffix)
            Path(save_dir).mkdir(parents=True, exist_ok=True)

        # 1. Performance Overview
        saved_files['performance_overview'] = self._plot_performance_overview(
            results, df, save_dir, show_plots
        )

        # 2. Trade Analysis
        if self.trades:
            saved_files['trade_analysis'] = self._plot_trade_analysis(
                results, save_dir, show_plots, path_suffix
            )

        # 3. Risk Analysis
        if self.equity_curve:
            saved_files['risk_analysis'] = self._plot_risk_analysis(
                results, save_dir, show_plots
            )

        # 4. Monthly Performance Heatmap
        if self.trades:
            saved_files['monthly_heatmap'] = self._plot_monthly_heatmap(
                results, save_dir, show_plots
            )

        # 5. Trade Distribution
        if self.trades:
            saved_files['trade_distribution'] = self._plot_trade_distribution(
                results, save_dir, show_plots
            )

        return saved_files

    def _plot_performance_overview(
        self,
        results: Dict,
        df: pd.DataFrame = None,
        save_dir: Optional[str] = None,
        show_plots: bool = True
    ) -> Optional[str]:
        """Create performance overview plot."""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Performance Overview', fontsize=16, fontweight='bold')

        # Equity curve
        if self.equity_curve:
            equity_df = pd.DataFrame(self.equity_curve)
            ax1 = axes[0, 0]
            ax1.plot(equity_df['timestamp'], equity_df['equity'],
                    label='Portfolio Value', linewidth=2, color='blue')
            ax1.axhline(y=self.initial_capital, color='gray', linestyle='--',
                       label='Initial Capital', alpha=0.7)

            # Add benchmark if available
            if df is not None and 'close' in df.columns:
                benchmark_value = self.initial_capital * (df['close'] / df['close'].iloc[0])
                ax1.plot(df.index if hasattr(df, 'index') else range(len(df)),
                        benchmark_value, label='Buy & Hold', alpha=0.7, color='orange')

            ax1.set_title('Equity Curve')
            ax1.set_ylabel('Portfolio Value ($)')
            ax1.legend()
            ax1.grid(True, alpha=0.3)

        # Drawdown
        if self.equity_curve:
            equity_df = pd.DataFrame(self.equity_curve)
            equity_df['peak'] = equity_df['equity'].cummax()
            equity_df['drawdown'] = (equity_df['equity'] - equity_df['peak']) / equity_df['peak'] * 100

            ax2 = axes[0, 1]
            ax2.fill_between(equity_df['timestamp'], 0, equity_df['drawdown'],
                           color='red', alpha=0.3)
            ax2.plot(equity_df['timestamp'], equity_df['drawdown'], color='red', linewidth=1)
            ax2.set_title('Drawdown')
            ax2.set_ylabel('Drawdown (%)')
            ax2.grid(True, alpha=0.3)

        # Returns distribution
        if self.equity_curve:
            equity_df = pd.DataFrame(self.equity_curve)
            returns = equity_df['equity'].pct_change().dropna()

            ax3 = axes[1, 0]
            ax3.hist(returns * 100, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
            ax3.axvline(returns.mean() * 100, color='red', linestyle='--',
                       label=f'Mean: {returns.mean()*100:.2f}%')
            ax3.set_title('Daily Returns Distribution')
            ax3.set_xlabel('Daily Return (%)')
            ax3.set_ylabel('Frequency')
            ax3.legend()
            ax3.grid(True, alpha=0.3)

        # Performance metrics summary
        ax4 = axes[1, 1]
        ax4.axis('off')

        metrics_text = f"""
        Performance Summary

        Total Return: {results.get('total_return_pct', 0):.2f}%
        Sharpe Ratio: {results.get('sharpe_ratio', 0):.2f}
        Max Drawdown: {results.get('max_drawdown', 0):.2f}%
        Win Rate: {results.get('win_rate', 0):.2f}%
        Total Trades: {results.get('total_trades', 0)}
        Profit Factor: {results.get('profit_factor', 0):.2f}
        """

        ax4.text(0.1, 0.9, metrics_text, transform=ax4.transAxes, fontsize=12,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightgray'))

        plt.tight_layout()

        file_path = None
        if save_dir:
            file_path = Path(save_dir) / 'performance_overview.png'
            plt.savefig(file_path, dpi=300, bbox_inches='tight')

        if show_plots:
            plt.show()
        else:
            plt.close()

        return str(file_path) if file_path else None

    def _plot_trade_analysis(
        self,
        results: Dict,
        save_dir: Optional[str] = None,
        show_plots: bool = True,
        path_suffix: Optional[str] = None
    ) -> Optional[str]:
        """Create trade analysis plots."""
        if not self.trades:
            return None

        trades_df = pd.DataFrame(self.trades)
        pnl_col = 'net_pnl' if 'net_pnl' in trades_df.columns else 'pnl'

        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Trade Analysis', fontsize=16, fontweight='bold')

        # Trade P&L over time
        ax1 = axes[0, 0]
        cumulative_pnl = trades_df[pnl_col].cumsum()
        ax1.plot(range(len(cumulative_pnl)), cumulative_pnl, marker='o', linewidth=2)
        ax1.set_title('Cumulative P&L by Trade')
        ax1.set_xlabel('Trade Number')
        ax1.set_ylabel('Cumulative P&L ($)')
        ax1.grid(True, alpha=0.3)

        # Win/Loss distribution
        ax2 = axes[0, 1]
        wins = trades_df[trades_df[pnl_col] > 0][pnl_col]
        losses = trades_df[trades_df[pnl_col] <= 0][pnl_col]

        ax2.hist([wins, losses], bins=20, label=['Wins', 'Losses'],
                color=['green', 'red'], alpha=0.7)
        ax2.set_title('Win/Loss Distribution')
        ax2.set_xlabel('P&L ($)')
        ax2.set_ylabel('Frequency')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # Trade duration analysis
        if 'bars_held' in trades_df.columns:
            ax3 = axes[1, 0]
            ax3.scatter(trades_df['bars_held'], trades_df[pnl_col], alpha=0.6)
            ax3.set_title('P&L vs Holding Period')
            ax3.set_xlabel('Bars Held')
            ax3.set_ylabel('P&L ($)')
            ax3.grid(True, alpha=0.3)

        # Monthly performance
        if 'entry_time' in trades_df.columns:
            ax4 = axes[1, 1]
            trades_df['month'] = pd.to_datetime(trades_df['entry_time']).dt.month
            monthly_pnl = trades_df.groupby('month')[pnl_col].sum()

            bars = ax4.bar(monthly_pnl.index, monthly_pnl.values,
                          color=['green' if x > 0 else 'red' for x in monthly_pnl.values])
            ax4.set_title('Monthly P&L')
            ax4.set_xlabel('Month')
            ax4.set_ylabel('P&L ($)')
            ax4.set_xticks(range(1, 13))
            ax4.grid(True, alpha=0.3)

        plt.tight_layout()

        file_path = None
        if save_dir:
            if path_suffix:
                save_dir = os.path.join(save_dir, path_suffix)
            file_path = Path(save_dir) / 'trade_analysis.png'
            plt.savefig(file_path, dpi=300, bbox_inches='tight')

        if show_plots:
            plt.show()
        else:
            plt.close()

        return str(file_path) if file_path else None

    def _plot_risk_analysis(
        self,
        results: Dict,
        save_dir: Optional[str] = None,
        show_plots: bool = True,
        path_suffix: Optional[str] = None
    ) -> Optional[str]:
        """Create risk analysis plots."""
        if not self.equity_curve:
            return None

        equity_df = pd.DataFrame(self.equity_curve)

        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Risk Analysis', fontsize=16, fontweight='bold')

        # Rolling Sharpe ratio
        equity_df['returns'] = equity_df['equity'].pct_change()
        window = min(30, len(equity_df) // 4)  # 30-day or 1/4 of data

        if window > 1:
            ax1 = axes[0, 0]
            rolling_sharpe = equity_df['returns'].rolling(window).mean() / equity_df['returns'].rolling(window).std() * np.sqrt(252)
            ax1.plot(equity_df['timestamp'], rolling_sharpe)
            ax1.set_title(f'Rolling Sharpe Ratio ({window}-period)')
            ax1.set_ylabel('Sharpe Ratio')
            ax1.grid(True, alpha=0.3)

        # Volatility analysis
        ax2 = axes[0, 1]
        rolling_vol = equity_df['returns'].rolling(window).std() * np.sqrt(252) * 100
        ax2.plot(equity_df['timestamp'], rolling_vol, color='orange')
        ax2.set_title(f'Rolling Volatility ({window}-period)')
        ax2.set_ylabel('Volatility (%)')
        ax2.grid(True, alpha=0.3)

        # Underwater curve (drawdown)
        equity_df['peak'] = equity_df['equity'].cummax()
        equity_df['drawdown'] = (equity_df['equity'] - equity_df['peak']) / equity_df['peak'] * 100

        ax3 = axes[1, 0]
        ax3.fill_between(equity_df['timestamp'], 0, equity_df['drawdown'],
                        color='red', alpha=0.3)
        ax3.plot(equity_df['timestamp'], equity_df['drawdown'], color='red')
        ax3.set_title('Underwater Curve')
        ax3.set_ylabel('Drawdown (%)')
        ax3.grid(True, alpha=0.3)

        # Risk-Return scatter
        ax4 = axes[1, 1]
        if len(equity_df) > 1:
            total_return = (equity_df['equity'].iloc[-1] / equity_df['equity'].iloc[0] - 1) * 100
            volatility = equity_df['returns'].std() * np.sqrt(252) * 100

            ax4.scatter(volatility, total_return, s=100, color='blue')
            ax4.set_xlabel('Volatility (%)')
            ax4.set_ylabel('Total Return (%)')
            ax4.set_title('Risk-Return Profile')
            ax4.grid(True, alpha=0.3)

        plt.tight_layout()

        file_path = None
        if save_dir:
            if path_suffix:
                save_dir = os.path.join(save_dir, path_suffix)
            file_path = Path(save_dir) / 'risk_analysis.png'
            plt.savefig(file_path, dpi=300, bbox_inches='tight')

        if show_plots:
            plt.show()
        else:
            plt.close()

        return str(file_path) if file_path else None

    def _plot_monthly_heatmap(
        self,
        results: Dict,
        save_dir: Optional[str] = None,
        show_plots: bool = True,
        path_suffix: Optional[str] = None
    ) -> Optional[str]:
        """Create monthly performance heatmap."""
        if not self.trades:
            return None

        trades_df = pd.DataFrame(self.trades)
        if 'entry_time' not in trades_df.columns:
            return None

        pnl_col = 'net_pnl' if 'net_pnl' in trades_df.columns else 'pnl'

        # Create monthly performance matrix
        trades_df['entry_time'] = pd.to_datetime(trades_df['entry_time'])
        trades_df['year'] = trades_df['entry_time'].dt.year
        trades_df['month'] = trades_df['entry_time'].dt.month

        monthly_returns = trades_df.groupby(['year', 'month'])[pnl_col].sum().reset_index()

        if len(monthly_returns) == 0:
            return None

        # Pivot for heatmap
        heatmap_data = monthly_returns.pivot(index='year', columns='month', values=pnl_col)

        fig, ax = plt.subplots(figsize=(12, 8))

        # Create heatmap
        sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='RdYlGn',
                   center=0, ax=ax, cbar_kws={'label': 'P&L ($)'})

        ax.set_title('Monthly Performance Heatmap', fontsize=14, fontweight='bold')
        ax.set_xlabel('Month')
        ax.set_ylabel('Year')

        # Set month labels - only for months that exist in the data
        month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                       'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
        # Get the actual months present in the data
        actual_months = sorted(heatmap_data.columns)
        actual_month_labels = [month_labels[m-1] for m in actual_months]
        ax.set_xticklabels(actual_month_labels)

        plt.tight_layout()

        file_path = None
        if save_dir:
            if path_suffix:
                save_dir = os.path.join(save_dir, path_suffix)
            file_path = Path(save_dir) / 'monthly_heatmap.png'
            plt.savefig(file_path, dpi=300, bbox_inches='tight')

        if show_plots:
            plt.show()
        else:
            plt.close()

        return str(file_path) if file_path else None

    def _plot_trade_distribution(
        self,
        results: Dict,
        save_dir: Optional[str] = None,
        show_plots: bool = True,
        path_suffix: Optional[str] = None
    ) -> Optional[str]:
        """Create trade distribution analysis."""
        if not self.trades:
            return None

        trades_df = pd.DataFrame(self.trades)
        pnl_col = 'net_pnl' if 'net_pnl' in trades_df.columns else 'pnl'

        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Trade Distribution Analysis', fontsize=16, fontweight='bold')

        # P&L distribution
        ax1 = axes[0, 0]
        ax1.hist(trades_df[pnl_col], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
        ax1.axvline(trades_df[pnl_col].mean(), color='red', linestyle='--',
                   label=f'Mean: ${trades_df[pnl_col].mean():.2f}')
        ax1.set_title('P&L Distribution')
        ax1.set_xlabel('P&L ($)')
        ax1.set_ylabel('Frequency')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # Box plot of wins vs losses
        ax2 = axes[0, 1]
        wins = trades_df[trades_df[pnl_col] > 0][pnl_col]
        losses = trades_df[trades_df[pnl_col] <= 0][pnl_col]

        box_data = [wins.values, losses.values]
        ax2.boxplot(box_data, labels=['Wins', 'Losses'])
        ax2.set_title('Win/Loss Box Plot')
        ax2.set_ylabel('P&L ($)')
        ax2.grid(True, alpha=0.3)

        # Cumulative distribution
        ax3 = axes[1, 0]
        sorted_pnl = np.sort(trades_df[pnl_col])
        cumulative_prob = np.arange(1, len(sorted_pnl) + 1) / len(sorted_pnl)
        ax3.plot(sorted_pnl, cumulative_prob, marker='o', markersize=3)
        ax3.set_title('Cumulative Distribution Function')
        ax3.set_xlabel('P&L ($)')
        ax3.set_ylabel('Cumulative Probability')
        ax3.grid(True, alpha=0.3)

        # Q-Q plot (normal distribution)
        ax4 = axes[1, 1]
        from scipy import stats
        stats.probplot(trades_df[pnl_col], dist="norm", plot=ax4)
        ax4.set_title('Q-Q Plot (Normal Distribution)')
        ax4.grid(True, alpha=0.3)

        plt.tight_layout()

        file_path = None
        if save_dir:
            if path_suffix:
                save_dir = os.path.join(save_dir, path_suffix)
            file_path = Path(save_dir) / 'trade_distribution.png'
            plt.savefig(file_path, dpi=300, bbox_inches='tight')

        if show_plots:
            plt.show()
        else:
            plt.close()

        return str(file_path) if file_path else None

    def export_results(
        self,
        results: Dict,
        export_dir: str,
        include_trades: bool = True,
        include_equity_curve: bool = True,
        include_report: bool = True,
        path_suffix: Optional[str] = None
    ) -> Dict[str, str]:
        """
        Export all results to files.

        Parameters:
        -----------
        results : Dict
            Backtest results
        export_dir : str
            Directory to export files
        include_trades : bool
            Whether to export trades data
        include_equity_curve : bool
            Whether to export equity curve data
        include_report : bool
            Whether to export text report

        Returns:
        --------
        exported_files : Dict[str, str]
            Dictionary of exported file types and paths
        """
        export_path = Path(export_dir)
        if path_suffix:
            export_path = os.path.join(export_dir, path_suffix)
        export_path.mkdir(parents=True, exist_ok=True)

        exported_files = {}

        # Export trades
        if include_trades and self.trades:
            trades_df = pd.DataFrame(self.trades)
            trades_file = export_path / 'trades.csv'
            trades_df.to_csv(trades_file, index=False)
            exported_files['trades'] = str(trades_file)

        # Export equity curve
        if include_equity_curve and self.equity_curve:
            equity_df = pd.DataFrame(self.equity_curve)
            equity_file = export_path / 'equity_curve.csv'
            equity_df.to_csv(equity_file, index=False)
            exported_files['equity_curve'] = str(equity_file)

        # Export results as JSON
        results_file = export_path / 'results.json'
        # Convert numpy types to native Python types for JSON serialization
        json_results = {}
        for key, value in results.items():
            if isinstance(value, (np.integer, np.floating)):
                json_results[key] = value.item()
            elif isinstance(value, np.ndarray):
                json_results[key] = value.tolist()
            elif pd.isna(value):
                json_results[key] = None
            else:
                json_results[key] = value

        with open(results_file, 'w') as f:
            json.dump(json_results, f, indent=2, default=str)
        exported_files['results'] = str(results_file)

        # Export text report
        if include_report:
            report_file = export_path / 'report.txt'
            report_text = self.generate_detailed_report(results)
            with open(report_file, 'w') as f:
                f.write(report_text)
            exported_files['report'] = str(report_file)

        return exported_files

    def print_summary(self, results: Dict):
        """
        Print a concise summary of results.

        Parameters:
        -----------
        results : Dict
            Backtest results
        """
        print("\n" + "="*60)
        print("BACKTEST SUMMARY")
        print("="*60)
        print(f"Total Return:     {results.get('total_return_pct', 0):>8.2f}%")
        print(f"Sharpe Ratio:     {results.get('sharpe_ratio', 0):>8.2f}")
        print(f"Max Drawdown:     {results.get('max_drawdown', 0):>8.2f}%")
        print(f"Win Rate:         {results.get('win_rate', 0):>8.2f}%")
        print(f"Total Trades:     {results.get('total_trades', 0):>8}")
        print(f"Profit Factor:    {results.get('profit_factor', 0):>8.2f}")
        print("="*60)


In [ ]:
class BacktestNoLib(BacktestBase):
    """
    Backtester that uses ML model predictions as trading signals.
    Supports trailing stop loss and various position sizing strategies.
    """

    def __init__(
        self,
        initial_capital: float = 10000.0,
        position_size: float = 1.0,
        trailing_stop_pct: float = 2.0,
        take_profit_pct: Optional[float] = None,
        commission: float = 0.001,
        slippage: float = 0.0005,
        use_probability_threshold: bool = True,
        probability_threshold: float = 0.6,
        max_holding_bars: Optional[int] = None
    ):
        """
        Initialize the ML Backtester.

        Parameters:
        -----------
        initial_capital : float
            Starting capital for backtesting
        position_size : float
            Fraction of capital to use per trade (0.0 to 1.0)
        trailing_stop_pct : float
            Trailing stop loss percentage (e.g., 2.0 for 2%)
        take_profit_pct : float, optional
            Take profit percentage (e.g., 5.0 for 5%)
        commission : float
            Commission per trade as a fraction (e.g., 0.001 for 0.1%)
        slippage : float
            Slippage per trade as a fraction (e.g., 0.0005 for 0.05%)
        use_probability_threshold : bool
            Whether to use probability threshold for entry
        probability_threshold : float
            Minimum probability to enter trade (0.0 to 1.0)
        max_holding_bars : int, optional
            Maximum number of bars to hold a position
        """
        # Initialize base class
        super().__init__(initial_capital=initial_capital, commission=commission, slippage=slippage)

        # BacktestNoLib specific parameters
        self.position_size = position_size
        self.trailing_stop_pct = trailing_stop_pct
        self.take_profit_pct = take_profit_pct
        self.use_probability_threshold = use_probability_threshold
        self.probability_threshold = probability_threshold
        self.max_holding_bars = max_holding_bars

        # Trading state
        self.capital = initial_capital
        self.position = 0  # Number of shares/units
        self.entry_price = 0
        self.highest_price = 0
        self.trailing_stop_price = 0
        self.bars_in_position = 0

    def reset(self):
        """Reset the backtester to initial state."""
        self.capital = self.initial_capital
        self.position = 0
        self.entry_price = 0
        self.highest_price = 0
        self.trailing_stop_price = 0
        self.bars_in_position = 0
        self.trades = []
        self.equity_curve = []
        self.signals = []

    def calculate_position_size(self, price: float) -> float:
        """
        Calculate the number of units to buy based on position sizing.

        Parameters:
        -----------
        price : float
            Current price

        Returns:
        --------
        units : float
            Number of units to buy
        """
        capital_to_use = self.capital * self.position_size
        units = capital_to_use / price
        return units

    def enter_long(self, price: float, timestamp: pd.Timestamp, signal_strength: float = 1.0):
        """
        Enter a long position.

        Parameters:
        -----------
        price : float
            Entry price
        timestamp : pd.Timestamp
            Entry timestamp
        signal_strength : float
            Strength of the signal (e.g., probability from model)
        """
        if self.position > 0:
            return  # Already in position

        print(f"\nBUY SIGNAL AT {price:.2f} ON {timestamp}")

        # Apply slippage
        entry_price_with_slippage = price * (1 + self.slippage)

        # Calculate position size
        units = self.calculate_position_size(entry_price_with_slippage)

        # Calculate commission
        commission_cost = units * entry_price_with_slippage * self.commission

        # Update state
        self.position = units
        self.entry_price = entry_price_with_slippage
        self.highest_price = entry_price_with_slippage
        self.trailing_stop_price = entry_price_with_slippage * (1 - self.trailing_stop_pct / 100)
        self.capital -= (units * entry_price_with_slippage + commission_cost)
        self.bars_in_position = 0

        # Record signal
        self.signals.append({
            'timestamp': timestamp,
            'type': 'ENTRY',
            'price': entry_price_with_slippage,
            'units': units,
            'signal_strength': signal_strength,
            'capital': self.capital
        })

    def exit_long(self, price: float, timestamp: pd.Timestamp, reason: str = 'SIGNAL'):
        """
        Exit a long position.

        Parameters:
        -----------
        price : float
            Exit price
        timestamp : pd.Timestamp
            Exit timestamp
        reason : str
            Reason for exit ('SIGNAL', 'TRAILING_STOP', 'TAKE_PROFIT', 'MAX_HOLDING')
        """
        if self.position == 0:
            return  # No position to exit

        print(f"\nSELL SIGNAL AT {price:.2f} ON {timestamp} ({self.bars_in_position} bars in position)")

        # Apply slippage
        exit_price_with_slippage = price * (1 - self.slippage)

        # Calculate commission
        commission_cost = self.position * exit_price_with_slippage * self.commission

        # Calculate P&L
        gross_pnl = self.position * (exit_price_with_slippage - self.entry_price)
        net_pnl = gross_pnl - commission_cost
        pnl_pct = (exit_price_with_slippage / self.entry_price - 1) * 100

        # Update capital
        self.capital += (self.position * exit_price_with_slippage - commission_cost)

        # Record trade
        self.trades.append({
            'entry_time': self.signals[-1]['timestamp'],
            'exit_time': timestamp,
            'entry_price': self.entry_price,
            'exit_price': exit_price_with_slippage,
            'units': self.position,
            'gross_pnl': gross_pnl,
            'net_pnl': net_pnl,
            'pnl_pct': pnl_pct,
            'bars_held': self.bars_in_position,
            'exit_reason': reason,
            'signal_strength': self.signals[-1]['signal_strength']
        })

        # Record signal
        self.signals.append({
            'timestamp': timestamp,
            'type': 'EXIT',
            'price': exit_price_with_slippage,
            'units': self.position,
            'reason': reason,
            'capital': self.capital,
            'pnl': net_pnl
        })

        # Reset position
        self.position = 0
        self.entry_price = 0
        self.highest_price = 0
        self.trailing_stop_price = 0
        self.bars_in_position = 0

    def update_trailing_stop(self, current_price: float):
        """
        Update trailing stop loss based on current price.

        Parameters:
        -----------
        current_price : float
            Current market price
        """
        if self.position == 0:
            return

        # Update highest price
        if current_price > self.highest_price:
            self.highest_price = current_price
            # Update trailing stop
            self.trailing_stop_price = self.highest_price * (1 - self.trailing_stop_pct / 100)

    def check_exit_conditions(self, current_price: float, timestamp: pd.Timestamp) -> bool:
        """
        Check if any exit conditions are met.

        Parameters:
        -----------
        current_price : float
            Current market price
        timestamp : pd.Timestamp
            Current timestamp

        Returns:
        --------
        exited : bool
            True if position was exited
        """
        if self.position == 0:
            return False

        # Check trailing stop
        if current_price <= self.trailing_stop_price:
            self.exit_long(current_price, timestamp, reason='TRAILING_STOP')
            return True

        # Check take profit
        if self.take_profit_pct is not None:
            take_profit_price = self.entry_price * (1 + self.take_profit_pct / 100)
            if current_price >= take_profit_price:
                self.exit_long(current_price, timestamp, reason='TAKE_PROFIT')
                return True

        # Check max holding period
        if self.max_holding_bars is not None:
            if self.bars_in_position >= self.max_holding_bars:
                self.exit_long(current_price, timestamp, reason='MAX_HOLDING')
                return True

        return False


    def run_backtest(
        self,
        df: pd.DataFrame,
        model,
        scaler,
        X_columns: List[str],
        probability_threshold = 0.6,
        trailing_stop_pct = 2.0,
        take_profit_pct = None,
        position_size_pct = 1.0,
        plot = False,  # Disable plotting to avoid memory issues
        printlog = False
    ) -> Tuple[Dict, pd.DataFrame]:
        """
        Run backtest using ML model predictions.

        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with OHLCV data and features
        model : sklearn model or similar
            Trained ML model with predict() and optionally predict_proba()
        scaler : sklearn scaler
            Fitted scaler for features
        X_columns : List[str]
            List of feature column names
        probability_threshold : float
            Minimum probability threshold for entry signals
        trailing_stop_pct : float
            Trailing stop loss percentage
        take_profit_pct : float, optional
            Take profit percentage
        position_size_pct : float
            Position size as percentage of capital
        plot : bool
            Whether to plot results (not implemented for BacktestNoLib)
        printlog : bool
            Whether to print log messages

        Returns:
        --------
        results : Dict
            Dictionary with backtest results and metrics
        trades : pd.DataFrame
            DataFrame with trade history
        """
        self.reset()

        close_column: str = 'close'
        timestamp_column: str = 'timestamp'

        # Ensure timestamp is datetime
        if timestamp_column in df.columns:
            df[timestamp_column] = pd.to_datetime(df[timestamp_column])
        else:
            df[timestamp_column] = pd.to_datetime(df.index)

        # Prepare features
        X = df[X_columns].values

        # Check if model expects fewer features (for Keras models)
        if hasattr(model, 'layers'):
            # Get expected input shape from model
            expected_features = model.input_shape[-1]  # Last dimension is features
            if X.shape[1] != expected_features:
                print(f"Warning: Feature mismatch. Data has {X.shape[1]} features, model expects {expected_features}. Using first {expected_features} features.")
                X = X[:, :expected_features]
        if scaler is not None:
            X_scaled = scaler.transform(X)
        else:
            X_scaled = X

        # Get predictions
        # Check if model is Keras/TensorFlow (CNN/LSTM) - needs 3D input
        if hasattr(model, 'layers'):  # Keras model
            # Get expected timesteps from model input shape
            expected_timesteps = model.input_shape[1]  # (None, timesteps, features)

            if expected_timesteps is None or expected_timesteps == 1:
                # Model accepts variable or single timestep
                X_input = X_scaled.reshape(X_scaled.shape[0], 1, X_scaled.shape[1])
            else:
                # Model expects specific sequence length - create rolling sequences
                print(f"Info: Model expects sequences of length {expected_timesteps}. Creating rolling sequences...")

                # Create sequences using rolling window
                sequences = []
                for i in range(len(X_scaled)):
                    if i < expected_timesteps - 1:
                        # Pad with first row for initial sequences
                        pad_length = expected_timesteps - i - 1
                        sequence = np.vstack([np.repeat([X_scaled[0]], pad_length, axis=0), X_scaled[:i+1]])
                    else:
                        # Use previous N rows as sequence
                        sequence = X_scaled[i - expected_timesteps + 1:i + 1]
                    sequences.append(sequence)

                X_input = np.array(sequences)
        else:
            X_input = X_scaled

        # Check if model is Keras/TensorFlow (CNN/LSTM) - needs 3D input
        if hasattr(model, 'layers'):  # Keras model
            # Get expected timesteps from model input shape
            expected_timesteps = model.input_shape[1]

            if expected_timesteps is None or expected_timesteps == 1:
                X_input = X_scaled.reshape(X_scaled.shape[0], 1, X_scaled.shape[1])
            else:
                # Create rolling sequences (already implemented)
                sequences = []
                for i in range(len(X_scaled)):
                    if i < expected_timesteps - 1:
                        pad_length = expected_timesteps - i - 1
                        sequence = np.vstack([np.repeat([X_scaled[0]], pad_length, axis=0), X_scaled[:i+1]])
                    else:
                        sequence = X_scaled[i - expected_timesteps + 1:i + 1]
                    sequences.append(sequence)
                X_input = np.array(sequences)
        else:
            # Traditional ML - needs flattened 2D input
            # First create sequences, then flatten
            expected_timesteps = 60  # Or get from training config
            sequences = []
            for i in range(len(X_scaled)):
                if i < expected_timesteps - 1:
                    pad_length = expected_timesteps - i - 1
                    sequence = np.vstack([np.repeat([X_scaled[0]], pad_length, axis=0), X_scaled[:i+1]])
                else:
                    sequence = X_scaled[i - expected_timesteps + 1:i + 1]
                sequences.append(sequence)

            # Flatten sequences: (samples, timesteps, features) -> (samples, timesteps*features)
            X_input = np.array(sequences).reshape(len(sequences), -1)

        predictions_raw = model.predict(X_input)

        # Get probabilities if available
        if hasattr(model, 'predict_proba'):
            # Sklearn models
            proba_array = model.predict_proba(X_input)
            predictions = predictions_raw  # Already class labels
            # For 3-class: [Down, Neutral, Up] -> use class 2 (Up)
            # For 2-class: [Down, Up] -> use class 1 (Up)
            up_class_idx = proba_array.shape[1] - 1  # Last class is always "Up"
            probabilities = proba_array[:, up_class_idx]
        elif hasattr(model, 'layers'):
            # Keras models - predict() returns probabilities
            proba_array = predictions_raw
            # Convert probabilities to class labels
            if len(proba_array.shape) > 1 and proba_array.shape[1] > 1:
                predictions = np.argmax(proba_array, axis=1)
                up_class_idx = proba_array.shape[1] - 1  # Last class is always "Up"
                probabilities = proba_array[:, up_class_idx]
            else:
                predictions = (proba_array > 0.5).astype(int).flatten()
                probabilities = proba_array.flatten()
        elif hasattr(model, 'decision_function'):
            # For models with decision_function, normalize to 0-1
            predictions = predictions_raw
            decision_scores = model.decision_function(X_input)
            probabilities = (decision_scores - decision_scores.min()) / (decision_scores.max() - decision_scores.min())
        else:
            # Default case
            predictions = predictions_raw
            probabilities = predictions.astype(float)

        # Run backtest
        for i in tqdm(range(len(df))):
            current_price = df[close_column].iloc[i]
            timestamp = df[timestamp_column].iloc[i]
            prediction = predictions[i]
            probability = probabilities[i]

            # Update bars in position
            if self.position > 0:
                self.bars_in_position += 1

                # Update trailing stop
                self.update_trailing_stop(current_price)

                # Check exit conditions
                if self.check_exit_conditions(current_price, timestamp):
                    continue

            # Check entry signal (class 2 for 3-class, class 1 for 2-class)
            # Assuming last class is always "Up"
            num_classes = len(np.unique(predictions))
            up_class = num_classes - 1  # Last class is "Up"

            if self.position == 0 and prediction == up_class:
                # Check probability threshold
                if self.use_probability_threshold:
                    if probability >= self.probability_threshold:
                        self.enter_long(current_price, timestamp, signal_strength=probability)
                else:
                    self.enter_long(current_price, timestamp, signal_strength=probability)

            # Record equity
            if self.position > 0:
                position_value = self.position * current_price
                total_equity = self.capital + position_value
            else:
                total_equity = self.capital

            self.equity_curve.append({
                'timestamp': timestamp,
                'equity': total_equity,
                'capital': self.capital,
                'position_value': self.position * current_price if self.position > 0 else 0,
                'in_position': self.position > 0
            })

        # Close any open position at the end
        if self.position > 0:
            final_price = df[close_column].iloc[-1]
            final_timestamp = df[timestamp_column].iloc[-1]
            self.exit_long(final_price, final_timestamp, reason='END_OF_DATA')

        # Calculate comprehensive metrics using base class
        results = self.calculate_comprehensive_metrics(df)

        # Create trades dataframe
        trades_df = pd.DataFrame(self.trades) if len(self.trades) > 0 else pd.DataFrame()

        return results, trades_df

    def calculate_metrics(self, df: pd.DataFrame, close_column: str = 'close') -> Dict:
        """
        Calculate performance metrics.

        Parameters:
        -----------
        df : pd.DataFrame
            Original dataframe with price data
        close_column : str
            Name of the close price column

        Returns:
        --------
        metrics : Dict
            Dictionary with performance metrics
        """
        if len(self.trades) == 0:
            # Calculate buy and hold return even with no trades
            buy_and_hold_return_pct = (df[close_column].iloc[-1] / df[close_column].iloc[0] - 1) * 100 if len(df) > 0 else 0

            return {
                'initial_capital': self.initial_capital,
                'final_value': self.capital,
                'total_return': 0,
                'total_return_pct': 0,
                'sharpe_ratio': 0,
                'max_drawdown': 0,
                'total_trades': 0,
                'won_trades': 0,
                'lost_trades': 0,
                'win_rate': 0,
                'avg_win': 0,
                'avg_loss': 0,
                'best_trade': 0,
                'worst_trade': 0,
                'profit_factor': 0,
                'buy_and_hold_return_pct': buy_and_hold_return_pct,
                'avg_bars_held': 0,
                'trades': [],
                'equity_curve': pd.DataFrame(self.equity_curve),
                'signals': self.signals
            }

        trades_df = pd.DataFrame(self.trades)
        equity_df = pd.DataFrame(self.equity_curve)

        # Basic metrics
        total_trades = len(self.trades)
        final_capital = self.capital
        total_return = final_capital - self.initial_capital
        total_return_pct = (final_capital / self.initial_capital - 1) * 100

        # Buy and hold return
        buy_and_hold_return_pct = (df[close_column].iloc[-1] / df[close_column].iloc[0] - 1) * 100

        # Win rate
        winning_trades = trades_df[trades_df['net_pnl'] > 0]
        losing_trades = trades_df[trades_df['net_pnl'] <= 0]
        win_rate = len(winning_trades) / total_trades * 100 if total_trades > 0 else 0

        # Average win/loss
        avg_win = winning_trades['net_pnl'].mean() if len(winning_trades) > 0 else 0
        avg_loss = losing_trades['net_pnl'].mean() if len(losing_trades) > 0 else 0

        # Profit factor
        gross_profit = winning_trades['net_pnl'].sum() if len(winning_trades) > 0 else 0
        gross_loss = abs(losing_trades['net_pnl'].sum()) if len(losing_trades) > 0 else 0
        profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')

        # Max drawdown
        equity_df['peak'] = equity_df['equity'].cummax()
        equity_df['drawdown'] = (equity_df['equity'] - equity_df['peak']) / equity_df['peak'] * 100
        max_drawdown = equity_df['drawdown'].min()

        # Sharpe ratio (annualized, assuming daily data)
        equity_df['returns'] = equity_df['equity'].pct_change()
        sharpe_ratio = equity_df['returns'].mean() / equity_df['returns'].std() * np.sqrt(252) if equity_df['returns'].std() > 0 else 0

        metrics = {
            'initial_capital': self.initial_capital,
            'final_value': final_capital,
            'total_return': total_return,
            'total_return_pct': total_return_pct,
            'sharpe_ratio': sharpe_ratio,
            'max_drawdown': max_drawdown,
            'total_trades': total_trades,
            'won_trades': len(winning_trades),
            'lost_trades': len(losing_trades),
            'win_rate': win_rate,
            'avg_win': avg_win,
            'avg_loss': avg_loss,
            'best_trade': winning_trades['net_pnl'].max() if len(winning_trades) > 0 else 0,
            'worst_trade': losing_trades['net_pnl'].min() if len(losing_trades) > 0 else 0,
            'profit_factor': profit_factor,
            'buy_and_hold_return_pct': buy_and_hold_return_pct,
            'avg_bars_held': trades_df['bars_held'].mean() if len(trades_df) > 0 else 0,
            'trades': self.trades,
            'equity_curve': equity_df,
            'signals': self.signals
        }

        return metrics

    def print_results(self, results: Dict):
        """
        Print formatted backtest results.

        Parameters:
        -----------
        results : Dict
            Results dictionary from run_backtest()
        """
        print("\n" + "="*80)
        print("BACKTEST RESULTS")
        print("="*80)

        print(f"\nCapital:")
        if 'initial_capital' in results:
            print(f"  Initial Capital:        ${results['initial_capital']:,.2f}")
        if 'final_value' in results:
            print(f"  Final Value:            ${results['final_value']:,.2f}")
        if 'total_return' in results:
            print(f"  Total Return:           ${results['total_return']:,.2f}")
        if 'total_return_pct' in results:
            print(f"  Total Return %:         {results['total_return_pct']:.2f}%")
        if 'buy_and_hold_return_pct' in results:
            print(f"  Buy & Hold Return %:    {results['buy_and_hold_return_pct']:.2f}%")

        print(f"\nTrades:")
        print(f"  Total Trades:           {results['total_trades']}")
        if 'won_trades' in results:
            print(f"  Won Trades:             {results['won_trades']}")
        if 'lost_trades' in results:
            print(f"  Lost Trades:            {results['lost_trades']}")
        if 'win_rate' in results:
            print(f"  Win Rate:               {results['win_rate']:.2f}%")

        print(f"\nProfit/Loss:")
        if 'avg_win' in results:
            print(f"  Average Win:            ${results['avg_win']:,.2f}")
        if 'avg_loss' in results:
            print(f"  Average Loss:           ${results['avg_loss']:,.2f}")
        if 'profit_factor' in results:
            print(f"  Profit Factor:          {results['profit_factor']:.2f}")

        print(f"\nRisk Metrics:")
        if 'max_drawdown' in results:
            print(f"  Max Drawdown:           {results['max_drawdown']:.2f}%")
        if 'sharpe_ratio' in results:
            print(f"  Sharpe Ratio:           {results['sharpe_ratio']:.2f}")

        print(f"\nHolding Period:")
        if 'avg_bars_held' in results:
            print(f"  Avg Bars Held:          {results['avg_bars_held']:.1f}")

        print(f"\nStrategy Parameters:")
        print(f"  Position Size:          {self.position_size * 100:.1f}%")
        print(f"  Trailing Stop:          {self.trailing_stop_pct:.2f}%")
        if self.take_profit_pct:
            print(f"  Take Profit:            {self.take_profit_pct:.2f}%")
        if self.use_probability_threshold:
            print(f"  Probability Threshold:  {self.probability_threshold:.2f}")
        if self.max_holding_bars:
            print(f"  Max Holding Bars:       {self.max_holding_bars}")

        print("\n" + "="*80)

        # Print last 10 trades
        if len(results['trades']) > 0:
            print("\nLast 10 Trades:")
            print("-"*80)
            trades_df = pd.DataFrame(results['trades'])
            print(trades_df[['entry_time', 'exit_time', 'entry_price', 'exit_price',
                           'net_pnl', 'pnl_pct', 'exit_reason']].tail(10).to_string(index=False))

    def plot_results(self, results: Dict, df: pd.DataFrame, close_column: str = 'close',
                    timestamp_column: str = 'Timestamp', save_path: Optional[str] = None):
        """
        Plot backtest results.

        Parameters:
        -----------
        results : Dict
            Results dictionary from run_backtest()
        df : pd.DataFrame
            Original dataframe with price data
        close_column : str
            Name of the close price column
        timestamp_column : str
            Name of the timestamp column
        save_path : str, optional
            Path to save the plot
        """
        fig, axes = plt.subplots(3, 1, figsize=(15, 12))

        # Ensure timestamp is datetime
        if timestamp_column in df.columns:
            df[timestamp_column] = pd.to_datetime(df[timestamp_column])
        else:
            df[timestamp_column] = pd.to_datetime(df.index)

        equity_df = results['equity_curve']

        # Plot 1: Price with Entry/Exit signals
        ax1 = axes[0]
        ax1.plot(df[timestamp_column], df[close_column], label='Price', color='black', alpha=0.7)

        # Plot entry signals
        entry_signals = [s for s in results['signals'] if s['type'] == 'ENTRY']
        if entry_signals:
            entry_times = [s['timestamp'] for s in entry_signals]
            entry_prices = [s['price'] for s in entry_signals]
            ax1.scatter(entry_times, entry_prices, color='green', marker='^', s=100,
                       label='Entry', zorder=5)

        # Plot exit signals
        exit_signals = [s for s in results['signals'] if s['type'] == 'EXIT']
        if exit_signals:
            exit_times = [s['timestamp'] for s in exit_signals]
            exit_prices = [s['price'] for s in exit_signals]
            exit_colors = ['red' if s['reason'] == 'TRAILING_STOP' else 'orange'
                          for s in exit_signals]
            ax1.scatter(exit_times, exit_prices, color=exit_colors, marker='v', s=100,
                       label='Exit', zorder=5)

        ax1.set_title('Price Chart with Entry/Exit Signals', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Date')
        ax1.set_ylabel('Price')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # Plot 2: Equity Curve
        ax2 = axes[1]
        ax2.plot(equity_df['timestamp'], equity_df['equity'], label='Portfolio Value',
                color='blue', linewidth=2)
        ax2.axhline(y=self.initial_capital, color='gray', linestyle='--',
                   label='Initial Capital', alpha=0.7)

        # Shade periods in position
        in_position = equity_df['in_position']
        ax2.fill_between(equity_df['timestamp'], equity_df['equity'].min(),
                        equity_df['equity'].max(), where=in_position,
                        alpha=0.1, color='green', label='In Position')

        ax2.set_title('Equity Curve', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Date')
        ax2.set_ylabel('Portfolio Value ($)')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # Plot 3: Drawdown
        ax3 = axes[2]
        ax3.fill_between(equity_df['timestamp'], 0, equity_df['drawdown'],
                        color='red', alpha=0.3)
        ax3.plot(equity_df['timestamp'], equity_df['drawdown'], color='red', linewidth=1)
        ax3.set_title('Drawdown', fontsize=14, fontweight='bold')
        ax3.set_xlabel('Date')
        ax3.set_ylabel('Drawdown (%)')
        ax3.grid(True, alpha=0.3)

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"\nPlot saved to: {save_path}")

        plt.show()


# Prepare data, train and test models

## Prepare data and models

In [ ]:
# Step 1: Download and prepare data
data_dict = download_and_prepare_data(CONFIG)

# Step 2: Create sequences and split data
train_val_test_data = prepare_train_val_test_data(data_dict, CONFIG)

In [ ]:
# Step 3: Create models
input_shape = (CONFIG['sequence_length'], data_dict['features'].shape[1])
models = create_models(input_shape, num_classes=3)

## Train models

In [ ]:
# Step 4: Train models
histories = train_models(models, train_val_test_data, CONFIG)

In [ ]:
save_models(models, data_dict['generated_results']['scaler'])

## Test models

In [ ]:
# Step 5: Test models
results = test_models(models, train_val_test_data)

## Visualization

In [ ]:
# Step 6: Visualize results
visualize_results(histories, results, models)

# BACKTEST MODELS

In [ ]:
backtester = BacktestNoLib()
X_train, y_train, X_val, y_val, X_test, y_test = train_val_test_data.values()
scaler = data_dict['generated_results']['scaler']
df = data_dict['generated_results']['df']
columns_to_exclude = ['timestamp', 'close', 'high', 'low', 'open', 'volume', 'future_close']
feature_names = data_dict['generated_results']['feature_names']
top_features = [col for col in df.columns if col not in columns_to_exclude and col in data_dict['feature_names']]
#
all_results = {}
all_equity_curves = {}


for model_name, model in models.items():
    results, trades = backtester.run_backtest(
            df = df,
            model = model,
            scaler = None,
            X_columns = top_features,
            probability_threshold = 0.4,
            trailing_stop_pct = 2.0,
            take_profit_pct = None,
            position_size_pct = 0.9,
            plot = False,
            printlog = True
        )

    # Add model name and trades to results
    results['model_name'] = model_name
    results['trades'] = trades

    # ✨ CREATE EQUITY CURVE FROM TRADES ✨
    if len(trades) > 0 and 'exit_capital' in trades.columns:
        # Create equity curve from trades
        equity_data = []
        for idx, trade in trades.iterrows():
            equity_data.append({
                'timestamp': trade['exit_time'],
                'equity': trade['exit_capital']
            })
        results['equity_curve'] = pd.DataFrame(equity_data)
    else:
        # No trades or missing data - create flat line
        initial = results.get('initial_capital', 100000)
        final = results.get('final_capital', initial)
        results['equity_curve'] = pd.DataFrame({
            'timestamp': [df['timestamp'].iloc[0], df['timestamp'].iloc[-1]],
            'equity': [initial, final]
        })

    # Store results
    all_results[model_name] = results

    # Store equity curve with validation
    if 'equity_curve' in results:
        equity_curve = results['equity_curve']
        print(f"  Debug - Equity curve type: {type(equity_curve)}")
        if equity_curve is not None:
            if hasattr(equity_curve, '__len__'):
                print(f"  Debug - Equity curve length: {len(equity_curve)}")
                if len(equity_curve) > 0:
                    all_equity_curves[model_name] = equity_curve
                    print(f"  ✓ Stored equity curve: {len(equity_curve)} points")
                else:
                    print(f"  ⚠ Equity curve is empty for {model_name}")
            else:
                print(f"  ⚠ Equity curve has no length attribute")
        else:
            print(f"  ⚠ Equity curve is None for {model_name}")
    else:
        print(f"  ⚠ No equity_curve in results for {model_name}")

    # Print results
    print(f"\nResults for {model_name}:")
    print(f"  Final Value: ${results['final_value']:,.2f}")
    print(f"  Total Return: {results['total_return_pct']:.2f}%")
    print(f"  Total Trades: {results['total_trades']}")
    print(f"  Win Rate: {results['win_rate']:.2f}%")
    print(f"  Sharpe Ratio: {results.get('sharpe_ratio', 0):.2f}")
    print(f"  Max Drawdown: {results.get('max_drawdown', 0):.2f}%")

    if results['total_trades'] > 0:
        backtester.create_comprehensive_visualizations(results,
                                    df=df,
                                    show_plots=True,
                                    model_name=model_name)

        # Create OHLC chart with trades
        _create_ohlc_with_trades_plot(df, trades, model_name)


In [ ]:
_create_models_comparison_plot(all_results, all_equity_curves, None)

In [ ]:
import os

folder = 'backtest_results'
for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    if os.path.isfile(file_path):
        os.remove(file_path)